# SINCON 2026 재현 노트북

인천 도시철도 역×날짜×시간대별 **승차 인원(boardings)**을 예측합니다.
공식 평가지표는 **RMSLE**이며, `log1p(boardings)`의 MSE를 학습합니다.
2026-07-01~2026-10-31의 123일, **전체 test ID 157,194개**가 제출 대상입니다.
로컬 `sample_submission.csv`는 92일만 포함한 오래된 양식이므로 제출 범위를 결정하는 데 쓰지 않습니다.

이 파일 하나에 프로젝트 코드·테스트·설정·의존성 목록을 포함했습니다.
경쟁 데이터, 사전학습 가중치, 계정 인증정보는 포함하지 않았습니다.
공식 과제·규정은 사용자가 제공한 [대회 안내](https://www.kaggle.com/competitions/sincon-2026/overview)를 기준으로 반영했습니다.

- Kaggle: 대회 데이터 4개 CSV를 Add Input으로 연결한 뒤 순서대로 실행합니다.
- Colab: 데이터 폴더를 준비하고 아래 `DATA_DIR_OVERRIDE`를 지정합니다.
- 모델은 기본 CPU에서 실행합니다. LightGBM Large·XGBoost·CatBoost를 개발 CV로 결합합니다. 마지막 fold는 재사용 검증 구간입니다.
- 자동 제출은 하지 않습니다. 마지막 셀에서 검수한 `submission.csv`를 제공합니다.

소스 스냅샷 SHA-256: `f1de8cacd1f5287a543cf9bd2fba1e484de1ea76e9331e07bb0851d240ae1468`


## 1. 코드 복원

기본 작업 경로는 Kaggle의 `/kaggle/working/SINCON`, Colab의 `/content/SINCON`입니다.
아래 셀을 다시 실행하면 포함된 소스·테스트·설정 파일을 현재 스냅샷으로 복원합니다.
데이터 파일은 복사하거나 변경하지 않습니다. 경로를 바꾸려면 `PROJECT_ROOT_OVERRIDE`만 수정합니다.


In [ ]:
from pathlib import Path
import csv
import hashlib
import json
import os
import shutil
import subprocess
import sys

PROJECT_ROOT_OVERRIDE = ""  # Optional absolute path for a different runtime.
IS_KAGGLE = Path("/kaggle/working").is_dir()
PROJECT_ROOT = Path(PROJECT_ROOT_OVERRIDE) if PROJECT_ROOT_OVERRIDE else Path("/kaggle/working/SINCON" if IS_KAGGLE else "/content/SINCON")
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)

EMBEDDED_FILES = {
    'src/__init__.py': '"""SINCON reproducible forecasting pipeline."""\n',
    'src/eda.py': '"""Reproducible, read-only SINCON CSV exploratory analysis.\n\nUsage: python src/eda.py --data-dir /path/to/sincon-2026 --out-dir reports\nRequires numpy and pandas; the compact SVG chart uses only the standard library.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport html\nimport json\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\n\ndef native(value):\n    if isinstance(value, dict):\n        return {str(k): native(v) for k, v in value.items()}\n    if isinstance(value, (list, tuple)):\n        return [native(v) for v in value]\n    if isinstance(value, (np.integer,)):\n        return int(value)\n    if isinstance(value, (np.floating,)):\n        return float(value) if np.isfinite(value) else None\n    if isinstance(value, pd.Timestamp):\n        return value.strftime(\'%Y-%m-%d\')\n    if pd.isna(value):\n        return None\n    return value\n\n\ndef table(frame, limit=20):\n    f = frame.head(limit).copy()\n    rows = [\'| \' + \' | \'.join(map(str, f.columns)) + \' |\',\n            \'| \' + \' | \'.join([\'---\'] * len(f.columns)) + \' |\']\n    for record in f.itertuples(index=False, name=None):\n        vals = []\n        for v in record:\n            if isinstance(v, (float, np.floating)):\n                vals.append(f\'{v:,.4f}\' if abs(v) < 10 else f\'{v:,.2f}\')\n            else:\n                vals.append(str(native(v)).replace(\'|\', \'\\\\|\'))\n        rows.append(\'| \' + \' | \'.join(vals) + \' |\')\n    return \'\\n\'.join(rows)\n\n\ndef draw_chart(path, series, title, ylabel):\n    """Small dependency-free SVG with explicit axes and all source points."""\n    width, height = 1100, 340\n    left, top, plot_w, plot_h = 82, 55, 980, 220\n    values = np.asarray(series.values, dtype=float)\n    ymax = max(float(np.nanmax(values)) * 1.08, 1.0)\n    xs = np.linspace(left, left + plot_w, len(values))\n    ys = top + plot_h - values / ymax * plot_h\n    points = \' \'.join(f\'{x:.2f},{y:.2f}\' for x, y in zip(xs, ys))\n    pieces = [f\'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" viewBox="0 0 {width} {height}">\',\n              \'<rect width="100%" height="100%" fill="#fff"/>\',\n              \'<g font-family="Arial, sans-serif" fill="#172b4d">\',\n              f\'<text x="{left}" y="28" font-size="20" font-weight="bold">{html.escape(title)}</text>\',\n              f\'<text x="{left}" y="46" font-size="12">{html.escape(ylabel)}</text>\']\n    for i in range(5):\n        v = ymax * i / 4\n        y = top + plot_h - plot_h * i / 4\n        pieces += [f\'<line x1="{left}" y1="{y}" x2="{left+plot_w}" y2="{y}" stroke="#e3e8ef"/>\',\n                   f\'<text x="{left-10}" y="{y+4}" font-size="12" text-anchor="end">{v:,.0f}</text>\']\n    pieces.append(f\'<polyline points="{points}" fill="none" stroke="#007e87" stroke-width="1.7"/>\')\n    for i in sorted(set(np.linspace(0, len(values)-1, min(7, len(values))).astype(int))):\n        label = series.index[i]\n        if isinstance(label, pd.Timestamp):\n            label = label.strftime(\'%Y-%m-%d\')\n        pieces.append(f\'<text x="{xs[i]}" y="{top+plot_h+25}" font-size="12" text-anchor="middle">{html.escape(str(label))}</text>\')\n    pieces += [\'<text x="82" y="325" font-size="11" fill="#56657a">Source: supplied train.csv; no target values from test.csv are available.</text>\', \'</g></svg>\']\n    path.write_text(\'\\n\'.join(pieces), encoding=\'utf-8\')\n\n\ndef analyze(data_dir: Path, out: Path):\n    out.mkdir(parents=True, exist_ok=True)\n    files = {name: data_dir / f\'{name}.csv\' for name in [\'train\', \'test\', \'metaData\', \'sample_submission\']}\n    train = pd.read_csv(files[\'train\'], parse_dates=[\'date\'], dtype={\'station_id\': \'int32\', \'hour\': \'int8\', \'boardings\': \'int32\', \'alightings\': \'int32\'})\n    test = pd.read_csv(files[\'test\'], parse_dates=[\'date\'], dtype={\'station_id\': \'int32\', \'hour\': \'int8\'})\n    meta = pd.read_csv(files[\'metaData\'], parse_dates=[\'first_date\', \'last_date\'], dtype={\'station_id\': \'int32\'})\n    sample = pd.read_csv(files[\'sample_submission\'])\n    frames = {\'train\': train, \'test\': test, \'metaData\': meta, \'sample_submission\': sample}\n    report = {\'source_directory\': str(data_dir.resolve()), \'files\': {}}\n    for name, frame in frames.items():\n        report[\'files\'][name] = {\n            \'rows\': len(frame), \'columns\': list(frame.columns), \'column_count\': frame.shape[1],\n            \'sha256\': hashlib.sha256(files[name].read_bytes()).hexdigest(),\n            \'bytes\': files[name].stat().st_size,\n            \'missing_cells\': frame.isna().sum().to_dict(),\n            \'memory_bytes\': int(frame.memory_usage(deep=True).sum()),\n            \'dtypes\': frame.dtypes.astype(str).to_dict(),\n        }\n    key = [\'date\', \'station_id\', \'hour\']\n    train = train.sort_values([\'station_id\', \'date\', \'hour\']).reset_index(drop=True)\n    hours = sorted(train.hour.unique().tolist())\n    station_coverage = train.groupby(\'station_id\').agg(\n        first_date=(\'date\', \'min\'), last_date=(\'date\', \'max\'),\n        observed_dates=(\'date\', \'nunique\'), rows=(\'date\', \'size\'))\n    station_coverage[\'calendar_dates\'] = (station_coverage.last_date - station_coverage.first_date).dt.days + 1\n    station_coverage[\'expected_rows_within_observed_lifetime\'] = station_coverage.calendar_dates * len(hours)\n    station_coverage[\'missing_rows_within_observed_lifetime\'] = station_coverage.expected_rows_within_observed_lifetime - station_coverage.rows\n    station_coverage = station_coverage.reset_index().merge(meta[[\'station_id\', \'station_name\', \'canonical_line\']], on=\'station_id\', validate=\'one_to_one\')\n    station_coverage.to_csv(out / \'station_coverage.csv\', index=False)\n    actual_metadata = station_coverage.merge(meta[[\'station_id\', \'first_date\', \'last_date\']], on=\'station_id\', suffixes=(\'_observed\', \'_metadata\'))\n    metadata_matches = bool(((actual_metadata.first_date_observed == actual_metadata.first_date_metadata) & (actual_metadata.last_date_observed == actual_metadata.last_date_metadata)).all())\n    train_timestamp = train.date + pd.to_timedelta(train.hour.astype(int), unit=\'h\')\n    delta = train_timestamp.groupby(train.station_id).diff().dt.total_seconds().dropna() / 3600\n    expected_grid = train.date.nunique() * train.station_id.nunique() * len(hours)\n    report[\'coverage\'] = {\n        \'train_start\': train.date.min(), \'train_end\': train.date.max(), \'train_days\': train.date.nunique(),\n        \'test_start\': test.date.min(), \'test_end\': test.date.max(), \'test_days\': test.date.nunique(),\n        \'station_count\': train.station_id.nunique(), \'hours\': hours,\n        \'train_duplicate_keys\': int(train.duplicated(key).sum()), \'test_duplicate_keys\': int(test.duplicated(key).sum()),\n        \'train_test_key_overlap\': len(train[key].merge(test[key], on=key)),\n        \'test_id_duplicates\': int(test.example_id.duplicated().sum()),\n        \'sample_id_duplicates\': int(sample.example_id.duplicated().sum()),\n        \'new_test_stations\': sorted(set(test.station_id) - set(train.station_id)),\n        \'metadata_unique_station_ids\': bool(meta.station_id.is_unique),\n        \'metadata_dates_match_observed\': metadata_matches,\n        \'global_full_grid_rows\': expected_grid,\n        \'global_grid_absent_rows\': expected_grid - len(train),\n        \'within_observed_lifetime_missing_rows\': station_coverage.missing_rows_within_observed_lifetime.sum(),\n        \'station_start_date_counts\': station_coverage.first_date.value_counts().sort_index().to_dict(),\n        \'observed_timestamp_gaps_hours\': delta.value_counts().sort_index().to_dict(),\n        \'test_complete_grid\': bool(len(test) == test.date.nunique() * test.station_id.nunique() * test.hour.nunique() and not test.duplicated(key).any()),\n        \'canonical_line_station_counts\': meta.canonical_line.value_counts().sort_index().to_dict(),\n    }\n    sample_keys = sample[[\'example_id\']].merge(test, on=\'example_id\', how=\'left\', validate=\'one_to_one\', indicator=True)\n    uncovered = test.loc[~test.example_id.isin(sample.example_id)]\n    report[\'submission_mismatch\'] = {\n        \'test_rows\': len(test), \'sample_rows\': len(sample), \'difference\': len(test)-len(sample),\n        \'sample_ids_absent_from_test\': int((sample_keys[\'_merge\'] != \'both\').sum()),\n        \'sample_period_start\': sample_keys.date.min(), \'sample_period_end\': sample_keys.date.max(),\n        \'sample_days\': sample_keys.date.nunique(),\n        \'uncovered_start\': uncovered.date.min(), \'uncovered_end\': uncovered.date.max(),\n        \'sample_matches_test_prefix_order\': bool(sample.example_id.tolist() == test.example_id.head(len(sample)).tolist()),\n        \'sample_values_all_zero\': bool((sample.boardings == 0).all()),\n        \'resolution\': \'User-supplied official rules confirm 157194 rows for July through October. Use full test IDs in their original order; the local 117576-row sample is stale/inconsistent and is not the final row-count authority.\'\n    }\n    for target in [\'boardings\', \'alightings\']:\n        y = train[target]\n        report[target] = {\n            \'min\': y.min(), \'max\': y.max(), \'mean\': y.mean(), \'std\': y.std(), \'median\': y.median(),\n            \'skewness\': y.skew(), \'negative_rows\': int((y < 0).sum()), \'zero_rows\': int((y == 0).sum()),\n            \'zero_fraction\': (y == 0).mean(), \'quantiles\': y.quantile([.01, .05, .25, .5, .75, .9, .95, .99, .999]).to_dict(),\n        }\n    report[\'same_row_boardings_alightings_correlation\'] = train.boardings.corr(train.alightings)\n    stats = train.groupby(\'station_id\').boardings.agg([\'size\', \'mean\', \'median\', \'std\', \'min\', \'max\']).reset_index().merge(meta[[\'station_id\',\'station_name\',\'canonical_line\']], on=\'station_id\')\n    stats[\'zero_fraction\'] = train.groupby(\'station_id\').boardings.apply(lambda y: (y == 0).mean()).reindex(stats.station_id).to_numpy()\n    stats.to_csv(out / \'station_target_stats.csv\', index=False)\n    report[\'highest_mean_stations\'] = stats.nlargest(10, \'mean\').to_dict(\'records\')\n    report[\'lowest_mean_stations\'] = stats.nsmallest(10, \'mean\').to_dict(\'records\')\n    train[\'weekday\'] = train.date.dt.dayofweek.astype(\'int8\')\n    train[\'month\'] = train.date.dt.strftime(\'%Y-%m\')\n    weekday = train.groupby(\'weekday\').boardings.agg([\'mean\',\'median\',\'size\']).reset_index()\n    hourly = train.groupby(\'hour\').boardings.mean()\n    monthly = train.groupby(\'month\').agg(rows=(\'boardings\',\'size\'), mean=(\'boardings\',\'mean\'), total=(\'boardings\',\'sum\')).reset_index()\n    daily = train.groupby(\'date\').boardings.sum()\n    daily_mean = train.groupby(\'date\').boardings.mean()\n    report[\'weekday_profiles\'] = weekday.to_dict(\'records\')\n    report[\'hour_mean_boardings\'] = hourly.to_dict()\n    report[\'monthly_profiles\'] = monthly.to_dict(\'records\')\n    report[\'aggregate_daily_autocorrelation\'] = {lag: daily.corr(daily.shift(lag)) for lag in [1, 7, 14, 28]}\n    weekday.to_csv(out / \'weekday_profile.csv\', index=False)\n    monthly.to_csv(out / \'monthly_profile.csv\', index=False)\n    daily.rename(\'boardings_total\').to_csv(out / \'daily_boardings.csv\')\n    maxima = train.nlargest(12,\'boardings\')[key+[\'boardings\',\'alightings\']].merge(meta[[\'station_id\',\'station_name\',\'canonical_line\']],on=\'station_id\')\n    report[\'largest_single_slot_counts\'] = maxima.to_dict(\'records\')\n    # Past-only weekly reference used strictly for an EDA spike flag, never as\n    # a whole-data training feature. A 3x / 1000 threshold is heuristic.\n    group = train.groupby([\'station_id\',\'hour\',\'weekday\'], sort=False).boardings\n    past = group.transform(lambda y: y.shift(1).rolling(8, min_periods=4).median())\n    ratios = train.boardings / past.replace(0,np.nan)\n    spikes = train.loc[(ratios > 3) & (train.boardings >= 1000), key+[\'boardings\']].copy()\n    spikes[\'past_8_same_weekday_median\'] = past.loc[spikes.index]\n    spikes[\'ratio\'] = ratios.loc[spikes.index]\n    spikes = spikes.merge(meta[[\'station_id\',\'station_name\']],on=\'station_id\').sort_values(\'ratio\',ascending=False)\n    spikes.to_csv(out / \'spike_candidates.csv\',index=False)\n    report[\'spike_heuristic\'] = {\'definition\':\'boardings >= 1000 and > 3x median of up to eight prior same-weekday observations at this station/hour; at least four prior observations\', \'rows\':len(spikes),\'top_rows\':spikes.head(15).to_dict(\'records\')}\n    # Synchrony is descriptive. A shared calendar/trend can explain correlations;\n    # no coordinates or physical adjacency are present in the input.\n    panel = train.pivot_table(index=\'date\',columns=\'station_id\',values=\'boardings\',aggfunc=\'sum\')\n    corr = panel.corr(min_periods=100)\n    residual = panel - panel.groupby(panel.index.dayofweek).transform(\'mean\')\n    adjusted = residual.corr(min_periods=100)\n    pairs=[]\n    names=meta.set_index(\'station_id\').station_name.to_dict()\n    lines=meta.set_index(\'station_id\').canonical_line.to_dict()\n    for a_i,a in enumerate(corr.columns):\n        for b in corr.columns[a_i+1:]:\n            pairs.append({\'station_a\':int(a),\'station_b\':int(b),\'name_a\':names[a],\'name_b\':names[b], \'same_canonical_line\':bool(lines[a]==lines[b]), \'raw_daily_correlation\':corr.loc[a,b], \'weekday_demeaned_correlation\':adjusted.loc[a,b]})\n    pair_df=pd.DataFrame(pairs).sort_values(\'raw_daily_correlation\',ascending=False)\n    pair_df.to_csv(out / \'station_correlations.csv\',index=False)\n    lag_rows=[]\n    for row in pair_df.head(10).itertuples(index=False):\n        a,b=row.station_a,row.station_b\n        lag_corr={str(lag):residual[a].corr(residual[b].shift(lag)) for lag in range(-3,4)}\n        lag_rows.append({\'station_a\':a,\'station_b\':b,\'name_a\':names[a],\'name_b\':names[b],\'daily_lag_correlations\':lag_corr, \'highest_correlation_lag_days\':int(max(lag_corr,key=lambda k: lag_corr[k]))})\n    report[\'spatial_and_synchrony\']={\n        \'coordinates_available\':False,\'adjacency_available\':False,\'canonical_lines\': sorted(meta.canonical_line.unique().tolist()),\n        \'pair_count\':len(pair_df), \'raw_pair_correlation_quantiles\':pair_df.raw_daily_correlation.quantile([0,.25,.5,.75,.9,1]).to_dict(),\n        \'weekday_demeaned_pair_correlation_quantiles\':pair_df.weekday_demeaned_correlation.quantile([0,.25,.5,.75,.9,1]).to_dict(),\n        \'top_pairs\':pair_df.head(10).to_dict(\'records\'), \'lag_diagnostics_top_10_pairs\':lag_rows,\n        \'interpretation\':\'Daily synchrony and coarse day-lag association only. Not a physical travel-time/propagation estimate. Station ID numeric proximity does not establish adjacency.\'}\n    report[\'feature_availability\']={\n        \'future_known_from_files\':[\'date\',\'station_id\',\'hour\',\'canonical_line\'],\n        \'train_only_observations\':[\'boardings\',\'alightings\'],\n        \'absent_from_files\':[\'weather\',\'holiday indicator\',\'discount/event indicator\',\'station coordinates\',\'physical adjacency\',\'service disruption labels\'],\n        \'calendar_derivable\':[\'weekday\',\'month\',\'day of year\',\'cyclic hour/weekday/day of year\'],\n        \'leakage_notes\':[\'No contemporaneous test alightings; use only values observed by forecast origin.\', \'Metadata last_date is a dataset coverage boundary, not a verified station closing date.\', \'No holdout target data may enter rolling means, scale normalization, correlations used as features or fitted calibration.\']}\n    report[\'official_task\'] = {\n        \'evidence\': \'User-supplied official Overview/Data/Rules; reports/sources/rules_user_provided.txt\',\n        \'metric\': \'RMSLE\', \'horizon_days\': 123, \'required_submission_rows\': 157194,\n        \'training_loss\': \'MSE on log1p(boardings); invert with expm1 then clip at zero\',\n        \'external_data_cutoff\': \'2026-06-30\',\n        \'holidays_allowed\': True, \'external_sources_and_license_record_required\': True,\n    }\n    (out/\'eda.json\').write_text(json.dumps(native(report),ensure_ascii=False,indent=2,allow_nan=False),encoding=\'utf-8\')\n    draw_chart(out/\'daily_boardings.svg\',daily,\'Daily boardings across observed stations\',\'Boardings / day (68 stations initially, 71 from 2025-06-28)\')\n    draw_chart(out/\'daily_mean_boardings.svg\',daily_mean,\'Daily mean boardings per station-hour\',\'Mean boardings / observed station-hour\')\n    draw_chart(out/\'hourly_boardings.svg\',hourly,\'Average boardings by hour\',\'Mean boardings / station-hour; all training dates\')\n    r=report\n    introduction=\'\'\'# SINCON 2026 데이터 EDA\n\n이 보고서는 제공된 CSV의 실제 값에서 재현한 관찰 결과다. 사용자가 제공한 [공식 Overview·Data·Rules 사본](sources/rules_user_provided.txt)에 따라 과제는 **인천 지하철 71개 역의 시간대별 승차 인원(boardings)을 2026-06-30 기준으로 7~10월 123일간 한 번에 예측하는 패널 시계열 회귀**다. 공식 지표는 **RMSLE**, 제출 행 수는 **157,194행**이다. `alightings`는 학습 데이터에만 있는 하차 인원 보조 변수다.\n\n## 1. 파일과 예측 범위\n\n\'\'\'\n    summary=pd.DataFrame([{\'파일\':f\'{name}.csv\',\'행\':len(df),\'열\':df.shape[1],\'셀 결측\':int(df.isna().sum().sum())} for name,df in frames.items()])\n    md=introduction+table(summary)+f\'\'\'\n\n- 학습: **2025-01-01~2026-06-30, {train.date.nunique()}일**.\n- 테스트: **2026-07-01~2026-10-31, {test.date.nunique()}일**, 71역 × 18시간 × 123일.\n- 제출 예시: 테스트 앞 **92일(2026-07-01~09-30)**의 ID 117,576개와 순서까지 일치한다.\n- 테스트에는 **10월 39,618행**이 추가로 있다. sample ID의 test 외 항목은 없고, 모든 ID와 (date, station_id, hour) 키는 고유하다.\n- **공식 규정은 157,194행을 요구하므로 전체 test의 123일을 예측한다.** 로컬 sample의 117,576행은 구버전/불일치로 기록하고, 최종 제출의 행 수는 공식 규정과 test ID를 기준으로 검수한다. 원본 sample은 수정하지 않는다.\n- `sample_submission.boardings`의 0은 양식 placeholder이며 실제 테스트 타겟이 아니다.\n\n## 2. 규칙성·결측·역별 관측 기간\n\n- 71개 역, 매일 **06~23시 18개 슬롯**이 제공된다. 각 역의 최초~최종 관측일 내부에는 누락 행이 **{int(station_coverage.missing_rows_within_observed_lifetime.sum())}개**, 중복 키도 0개다.\n- timestamp 차이는 **1시간**(같은 날) 또는 **7시간**(23시→다음 날 06시)뿐이다. 00~05시가 파일에 없다는 뜻이며 실제 운휴를 확인한 것은 아니다. 그 시간에 0을 채워 24시간 자료로 바꾸지 않는다.\n- 전체 71역 × 546일 × 18시간 격자에서 빈 {expected_grid-len(train):,}행은 3개 역의 첫 관측일 이전 구간이다. 이 값들은 결측 승객 수 또는 0명으로 취급하지 않는다.\n- 68개 역은 2025-01-01, 3개 역(검단호수공원·신검단중앙·아라)은 **2025-06-28**부터 관측된다. 테스트에는 완전히 새로운 station_id가 없다.\n- 메타데이터 `first_date`, `last_date`는 실제 관측 범위와 일치한다. 특히 `last_date=2026-06-30`은 데이터 종료일이지 폐역일이 아니다.\n- 노선별 역 수: {json.dumps(native(r[\'coverage\'][\'canonical_line_station_counts\']),ensure_ascii=False)}. 같은 역명이라도 다른 station_id를 자동 합치지 않는다.\n\n## 3. 타겟 분포와 급등\n\n\'\'\'\n    ys=r[\'boardings\']\n    md+=table(pd.DataFrame([{\'통계\':\'평균\',\'값\':ys[\'mean\']},{\'통계\':\'중앙값\',\'값\':ys[\'median\']},{\'통계\':\'표준편차\',\'값\':ys[\'std\']},{\'통계\':\'최솟값\',\'값\':ys[\'min\']},{\'통계\':\'최댓값\',\'값\':ys[\'max\']},{\'통계\':\'왜도\',\'값\':ys[\'skewness\']},{\'통계\':\'0인 행 수\',\'값\':ys[\'zero_rows\']},{\'통계\':\'0 비율\',\'값\':ys[\'zero_fraction\']}]))\n    md+=f\'\'\'\n\n우측 꼬리가 길다. 99% 분위수는 {ys[\'quantiles\'][.99]:,.0f}, 99.9% 분위수는 {ys[\'quantiles\'][.999]:,.2f}다. 음수는 없다. 공식 RMSLE에 맞춰 **log1p(boardings)의 MSE**를 기본 학습 손실로 사용한다. 예측은 expm1로 복원한 뒤 음수를 0으로 자른다. 원단위 MAE·RMSE·편향도 진단용으로 병행한다. 양수 카운트라는 사실만으로 Poisson 분포를 가정하지 않는다.\n\n최대 관측값:\n\n\'\'\'+table(maxima,12)+f\'\'\'\n\n동일 역·시간·요일의 과거 4~8주 중앙값 대비 3배 초과이면서 승차 1,000명 이상인 휴리스틱 급등 후보는 **{len(spikes):,}행**이다. 이벤트·오류·수송 변화 여부는 원자료 설명 없이는 확정할 수 없다. 무조건 삭제하거나 상한을 자르지 않는다. 상세 목록은 `spike_candidates.csv`에 있다.\n\n\'\'\'+table(spikes.head(10),10)+\'\'\'\n\n## 4. 달력 패턴과 시점 변화\n\n요일 0=월요일, 6=일요일이다. 다음 수치는 역·시간당 평균이며 총량과 구별한다.\n\n\'\'\'+table(weekday)+\'\'\'\n\n월별 표는 그 달에 관측된 행의 평균이다. 3개 역 추가에 따른 구성 변화와 달력 효과를 함께 고려해야 한다.\n\n\'\'\'+table(monthly,30)+f\'\'\'\n\n전체 일별 승차 총량의 자기상관: {json.dumps(native(r[\'aggregate_daily_autocorrelation\']),ensure_ascii=False)}. 공통 요일과 추세가 포함된 EDA 상관이며 모델 잔차 ACF가 아니다. CV 후에는 역·시간별 잔차를 날짜순으로 다시 평가한다.\n\n![일별 총승차](daily_boardings.svg)\n\n![역·시간당 일별 평균](daily_mean_boardings.svg)\n\n![시간대별 평균 승차](hourly_boardings.svg)\n\n## 5. 역 규모와 다개체 연결성\n\n평균 승차가 큰 역:\n\n\'\'\'+table(stats.nlargest(10,\'mean\')[[\'station_id\',\'station_name\',\'canonical_line\',\'mean\',\'median\',\'max\']])+\'\'\'\n\n제공 메타데이터에는 노선은 있으나 좌표·역 간 거리·연결 목록이 없다. ID의 숫자 차이로 거리나 인접 관계를 만들지 않는다. 같은 노선 구분과 역별 이력만 즉시 사용할 수 있다.\n\n일별 총량의 역 간 상관 상위쌍(최소 동시 관측 100일):\n\n\'\'\'+table(pair_df.head(10),10)+f\'\'\'\n\n- 전체 {len(pair_df):,}개 역 쌍의 원자료 상관 중앙값: {pair_df.raw_daily_correlation.median():.4f}.\n- 각 역의 요일 평균을 제거한 상관 중앙값: {pair_df.weekday_demeaned_correlation.median():.4f}.\n- 상위 10쌍의 ±3일 지연 상관은 `eda.json`에 기록했다. 이는 날짜 단위의 동조 점검이다. 1시간보다 짧은 열차 이동/승객 전파를 검증하지 못하며 인과적 연결 또는 물리적 전파 지연의 근거가 아니다.\n- 전체 학습 구간으로 만든 상관은 EDA 전용이다. 모델의 그룹/이웃 특징으로 쓰려면 각 CV origin 이전 이력에서만 다시 계산한다.\n\n## 6. 입력 가용성과 누수 방지\n\n| 구분 | 변수/처리 |\n| --- | --- |\n| 미래에도 주어짐 | date, station_id, hour, canonical_line |\n| 파일에서 직접 파생 가능 | 요일·주말·월·연중일·시간 및 달력 주기 sin/cos |\n| 학습에만 있음 | boardings, alightings |\n| 없음 | 날씨, 공휴일 라벨, 할인·행사, 운행 장애, 좌표, 물리적 인접 정보 |\n| 규정상 허용된 외부 입력 | 2026-06-30 이전 공개되고 비상업 이용이 허용된 공휴일·일정 등. 출처·조회일·라이선스 기록 필요 |\n\n공휴일은 `src/holiday_calendar.py`의 검증된 정적 달력에서 생성한다. 공식 출처와 공개시점은 [holiday_sources.md](holiday_sources.md)에 기록했다. 원본 CSV에는 공휴일이 없으며 외부 특징으로 추가한 것이다. 임시공휴일과 신규 지정일은 각 CV 예측 기준일에 알려진 정보만 활성화한다. 2026-05-01 노동절과 2026-07-17 제헌절을 포함하고, 2026-09-28은 공휴일로 표시하지 않는다.\n\n같은 행의 승차·하차 상관은 {r[\'same_row_boardings_alightings_correlation\']:.4f}다. **미래 alightings는 test에 없으므로 동시점 하차 값을 입력하면 누수/추론 불일치다.** origin 이전의 요약·지연 하차만 쓸 수 있다. 123일 전체를 한 번에 예측할 때 day 8의 실제 lag7처럼 origin 이후 관측을 참조해서는 안 된다. 직접 다중 horizon 특징이나 origin에 고정한 계절/이동 통계가 필요하다.\n\n## 7. 재현 및 보존\n\n```bash\npython src/eda.py --data-dir data/raw --out-dir reports\n```\n\n이 스크립트는 원본을 수정하지 않는다. `eda.json`에 입력 SHA-256, 자료형, 정확한 수치와 정의가 있고, 보조 CSV에 역별 범위·분포·상관·급등 후보가 있다. 그림은 집계값을 그대로 사용한다. 파일 크기는 약 24 MB이므로 이 단계의 원본만으로 29 GB RAM이 부족하지 않다. 다중 origin × horizon × 파생변수를 확장하면 메모리가 증가하므로 float32·정수 다운캐스팅, origin 배치 생성, 메모리 사용량 계측이 필요하다.\n\'\'\'\n    (out/\'EDA.md\').write_text(md,encoding=\'utf-8\')\n    print(json.dumps(native({\'train_rows\':len(train),\'test_rows\':len(test),\'sample_rows\':len(sample),\'missing_rows\':station_coverage.missing_rows_within_observed_lifetime.sum(),\'spikes\':len(spikes),\'out_dir\':str(out.resolve())}),ensure_ascii=False))\n\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'--data-dir\', type=Path, default=Path(__file__).resolve().parents[1]/\'data\'/\'raw\')\n    parser.add_argument(\'--out-dir\', type=Path, default=Path(__file__).resolve().parents[1]/\'reports\')\n    args=parser.parse_args()\n    analyze(args.data_dir,args.out_dir)\n',
    'src/ensemble.py': '"""Development-OOF convex blends: primary-constrained log-MSE or legacy greedy."""\nimport numpy as np\nimport pandas as pd\nfrom .metrics import score\n\ndef fit_ensemble(y,predictions,metric=\'rmsle\',correlation_cutoff=0.95,policy=None):\n    if policy and policy.get(\'method\')==\'primary_constrained_log_mse\':\n        if metric!=\'rmsle\':raise ValueError(\'Constrained log-MSE blending requires RMSLE\')\n        return fit_primary_ensemble(y,predictions,policy)\n    if policy:raise ValueError(f\'Unknown ensemble policy: {policy}\')\n    names=list(predictions)\n    scores={n:score(y,predictions[n],metric) for n in names}\n    space={n:np.log1p(predictions[n]) if metric==\'rmsle\' else np.asarray(predictions[n]) for n in names}\n    target=np.log1p(y) if metric==\'rmsle\' else y\n    invert=lambda values:np.maximum(0,np.expm1(values)) if metric==\'rmsle\' else np.maximum(0,values)\n    residual=pd.DataFrame({n:space[n]-target for n in names})\n    corr=residual.corr()\n    ranked=sorted(names,key=scores.get)\n    retained=[];excluded={}\n    for name in ranked:\n        redundant=[other for other in retained if np.array_equal(space[name],space[other]) or corr.loc[name,other]>correlation_cutoff]\n        if redundant:excluded[name]={\'reason\':\'residual_correlation_above_threshold\',\'with\':redundant}\n        else:retained.append(name)\n    weights={n:0.0 for n in retained};weights[retained[0]]=1.0\n    blend=space[retained[0]].copy();best=score(y,invert(blend),metric)\n    for step in [0.25,0.1,0.05,0.02,0.01]:\n        for _ in range(40):\n            options=[]\n            for name in retained:\n                proposed=(1-step)*blend+step*space[name]\n                options.append((score(y,invert(proposed),metric),name,proposed))\n            trial,name,proposed=min(options,key=lambda item:item[0])\n            if trial>=best-1e-8:break\n            weights={n:w*(1-step) for n,w in weights.items()};weights[name]+=step\n            blend=proposed;best=trial\n    # A scalar adjustment is estimated only on development OOF, audit is untouched.\n    bias=float(np.mean(target-blend)) if metric==\'rmsle\' else float(np.median(y-blend)) if metric==\'mae\' else 0.0\n    corrected=invert(blend+bias)\n    if score(y,corrected,metric)>=best:bias=0.0\n    return {\'weights\':weights,\'bias\':bias,\'bias_space\':\'log1p\' if metric==\'rmsle\' else \'raw\',\'blend_space\':\'log1p\' if metric==\'rmsle\' else \'raw\',\'retained\':retained,\'excluded\':excluded,\'individual_scores\':scores,\'development_score\':score(y,invert(blend+bias),metric),\'metric\':metric},corr\n\ndef fit_primary_ensemble(y,predictions,policy):\n    """Minimize development log-MSE with a fixed minimum primary-model share.\n\n    The caller supplies development OOF only. Correlation filtering and intercept\n    calibration are disabled; the primary model cannot be silently removed.\n    """\n    from scipy.optimize import minimize\n    names=list(predictions)\n    primary=policy[\'primary_model\']\n    minimum=float(policy[\'primary_min_weight\'])\n    if primary not in names:raise ValueError(\'Primary model missing from predictions\')\n    if not .5<=minimum<=1:raise ValueError(\'Primary minimum weight must be in [0.5, 1]\')\n    if policy.get(\'bias_correction\',False):raise ValueError(\'This policy disables bias correction\')\n    truth=np.asarray(y,dtype=np.float64)\n    if truth.ndim!=1 or len(truth)==0 or not np.isfinite(truth).all() or np.any(truth<0):\n        raise ValueError(\'Expected finite nonnegative one-dimensional targets\')\n    arrays=[np.asarray(predictions[name],dtype=np.float64) for name in names]\n    if any(p.shape!=truth.shape or not np.isfinite(p).all() or np.any(p<0) for p in arrays):\n        raise ValueError(\'Predictions must match targets and be finite and nonnegative\')\n    residual=np.column_stack([np.log1p(p)-np.log1p(truth) for p in arrays])\n    gram=residual.T@residual/len(truth)\n    ix=names.index(primary)\n    lower=np.zeros(len(names));lower[ix]=minimum\n    start=lower+(1-minimum)/len(names)\n    result=minimize(lambda w:float(w@gram@w),start,jac=lambda w:2*gram@w,\n                    method=\'SLSQP\',bounds=[(float(lo),1.) for lo in lower],\n                    constraints=[{\'type\':\'eq\',\'fun\':lambda w:w.sum()-1.,\n                                  \'jac\':lambda w:np.ones_like(w)}],\n                    options={\'ftol\':1e-13,\'maxiter\':500})\n    if not result.success:raise RuntimeError(f\'Ensemble optimization failed: {result.message}\')\n    w=np.asarray(result.x)\n    if np.any(w<lower-1e-10) or abs(w.sum()-1)>1e-10:\n        raise RuntimeError(\'Ensemble optimizer violated convex constraints\')\n    # SLSQP may return a boundary value a few ulps below its lower bound.\n    w=np.maximum(w,lower)\n    w[ix]+=1-w.sum()\n    scores={name:score(truth,p,\'rmsle\') for name,p in zip(names,arrays)}\n    plan={\'weights\':dict(zip(names,map(float,w))),\'bias\':0.,\'bias_space\':\'log1p\',\n          \'blend_space\':\'log1p\',\'retained\':names,\'excluded\':{},\n          \'individual_scores\':scores,\'metric\':\'rmsle\',\'policy\':dict(policy),\n          \'optimizer\':\'SLSQP: convex quadratic log-MSE\',\n          \'correlation_filter_applied\':False,\'bias_correction_applied\':False}\n    plan[\'development_score\']=score(truth,predict_ensemble(predictions,plan),\'rmsle\')\n    return plan,pd.DataFrame(residual,columns=names).corr()\n\ndef predict_ensemble(predictions,plan):\n    log_space=plan.get(\'blend_space\')==\'log1p\'\n    pred=sum(weight*(np.log1p(predictions[name]) if log_space else np.asarray(predictions[name])) for name,weight in plan[\'weights\'].items() if weight>0)\n    return np.maximum(0,np.expm1(pred+plan[\'bias\']) if log_space else pred+plan[\'bias\'])\n',
    'src/features.py': '"""Origin-frozen direct forecasting features. Never read labels after origin."""\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\nfrom .holiday_calendar import calendar_features\n\nKEYS = [\'station_id\', \'hour\']\n\ndef load_data(data_dir):\n    import polars as pl\n    root = Path(data_dir)\n    result = {}\n    for name in [\'train\', \'test\', \'sample_submission\', \'metaData\']:\n        lf = pl.scan_csv(root / f\'{name}.csv\')\n        schema = lf.collect_schema()\n        casts = [pl.col(c).cast(pl.Int32) for c in [\'station_id\', \'hour\', \'boardings\', \'alightings\', \'canonical_line\'] if c in schema]\n        df = pd.DataFrame(lf.with_columns(casts).collect().to_dict(as_series=False))\n        if \'date\' in df: df[\'date\'] = pd.to_datetime(df[\'date\'])\n        for c in [\'station_id\', \'hour\', \'boardings\', \'alightings\', \'canonical_line\']:\n            if c in df: df[c] = df[c].astype(\'int32\')\n        result[name] = df\n    tr, te = result[\'train\'], result[\'test\']\n    for name, frame in [(\'train\',tr),(\'test\',te)]:\n        if frame.duplicated([\'date\'] + KEYS).any(): raise ValueError(f\'{name}: duplicate keys\')\n        if frame.isna().any().any(): raise ValueError(f\'{name}: input cells missing; inspect first\')\n    if tr.date.max() >= te.date.min(): raise ValueError(\'Test is not strictly after training\')\n    if not np.isfinite(tr[[\'boardings\',\'alightings\']]).all().all() or (tr[[\'boardings\',\'alightings\']] < 0).any().any():\n        raise ValueError(\'Training counts must be finite and nonnegative\')\n    return result\n\ndef make_features(history, query, metadata, origin):\n    origin = pd.Timestamp(origin)\n    hist = history.loc[history.date <= origin].copy()\n    if hist.empty: raise ValueError(\'No history at this origin\')\n    if (query.date <= origin).any(): raise ValueError(\'Queries must follow origin\')\n    q = query[[\'date\'] + KEYS].reset_index(drop=True).copy()\n    q[\'_order\'] = np.arange(len(q))\n    q[\'dow\'] = q.date.dt.dayofweek.astype(\'int8\')\n    hist[\'dow\'] = hist.date.dt.dayofweek.astype(\'int8\')\n    q = q.merge(metadata[[\'station_id\',\'canonical_line\']],on=\'station_id\',how=\'left\',validate=\'many_to_one\')\n    if q.canonical_line.isna().any(): raise ValueError(\'Station absent from metadata\')\n    hist = hist.merge(metadata[[\'station_id\',\'canonical_line\']],on=\'station_id\',how=\'left\',validate=\'many_to_one\')\n    q[\'horizon\'] = (q.date - origin).dt.days\n    q[\'month\'] = q.date.dt.month\n    q[\'day_of_month\'] = q.date.dt.day\n    q[\'day_of_year\'] = q.date.dt.dayofyear\n    q[\'date_index\'] = (q.date - pd.Timestamp(\'2025-01-01\')).dt.days\n    q[\'weekend\'] = (q.dow >= 5).astype(\'int8\')\n    q = pd.concat([q,calendar_features(q.date,as_of=origin)],axis=1)\n    for col, period in [(\'hour\',24),(\'dow\',7),(\'day_of_year\',365.25)]:\n        for trig in [\'sin\',\'cos\']:\n            q[f\'{col}_{trig}\'] = getattr(np,trig)(2*np.pi*q[col]/period)\n    inception = hist.groupby(\'station_id\',as_index=False).date.min().rename(columns={\'date\':\'observed_first_date\'})\n    q = q.merge(inception,on=\'station_id\',how=\'left\',validate=\'many_to_one\')\n    q[\'history_days\'] = (origin - q.observed_first_date).dt.days + 1\n    q[\'cold_start\'] = q.observed_first_date.isna().astype(\'int8\')\n    q = q.drop(columns=\'observed_first_date\')\n    # All aggregations are computed only once per origin, never using query labels.\n    for days in [28,56,112]:\n        h = hist.loc[hist.date > origin - pd.Timedelta(days=days)]\n        stats = h.groupby(KEYS+[\'dow\']).agg(\n            **{f\'board_mean_{days}\':(\'boardings\',\'mean\'),\n               f\'board_median_{days}\':(\'boardings\',\'median\'),\n               f\'board_std_{days}\':(\'boardings\',\'std\'),\n               f\'alight_mean_{days}\':(\'alightings\',\'mean\'),\n               f\'n_{days}\':(\'boardings\',\'size\')}).reset_index()\n        q = q.merge(stats,on=KEYS+[\'dow\'],how=\'left\',validate=\'many_to_one\')\n    short = hist.loc[hist.date > origin - pd.Timedelta(days=28)]\n    stats = short.groupby(KEYS).agg(recent_mean=(\'boardings\',\'mean\'),recent_std=(\'boardings\',\'std\'),recent_alight=(\'alightings\',\'mean\')).reset_index()\n    q = q.merge(stats,on=KEYS,how=\'left\',validate=\'many_to_one\')\n    for keys,label in [([\'canonical_line\',\'hour\',\'dow\'],\'line\'),([\'hour\',\'dow\'],\'global\')]:\n        agg = short.groupby(keys,as_index=False).boardings.median().rename(columns={\'boardings\':f\'{label}_median\'})\n        q = q.merge(agg,on=keys,how=\'left\',validate=\'many_to_one\')\n    # Weekly seasonal copy for arbitrary horizon: repeat the last observed weekday.\n    q[\'_lookup_date\'] = origin - pd.to_timedelta((origin.dayofweek - q.dow)%7,unit=\'D\')\n    lookup = hist[[\'date\']+KEYS+[\'boardings\',\'alightings\']].rename(columns={\'date\':\'_lookup_date\',\'boardings\':\'seasonal7\',\'alightings\':\'seasonal7_alight\'})\n    q = q.merge(lookup,on=[\'_lookup_date\']+KEYS,how=\'left\',validate=\'many_to_one\').drop(columns=\'_lookup_date\')\n    for days in [1,7,28]:\n        snap = hist.loc[hist.date == origin - pd.Timedelta(days=days-1),KEYS+[\'boardings\']].rename(columns={\'boardings\':f\'origin_lag_{days}\'})\n        q = q.merge(snap,on=KEYS,how=\'left\',validate=\'many_to_one\')\n    q[\'_lookup_date\'] = q.date - pd.Timedelta(days=364)\n    # This explicit guard keeps the feature valid if a longer forecast is requested.\n    q.loc[q._lookup_date > origin,\'_lookup_date\'] = pd.NaT\n    lookup = hist[[\'date\']+KEYS+[\'boardings\']].rename(columns={\'date\':\'_lookup_date\',\'boardings\':\'year364\'})\n    q = q.merge(lookup,on=[\'_lookup_date\']+KEYS,how=\'left\',validate=\'many_to_one\').drop(columns=\'_lookup_date\')\n    q[\'trend_28_112\'] = q.board_mean_28 / q.board_mean_112.clip(lower=1)\n    q[\'boarding_alighting_ratio\'] = q.board_mean_56 / q.alight_mean_56.clip(lower=1)\n    q[\'baseline\'] = q.board_median_56.fillna(q.line_median).fillna(q.global_median).fillna(float(hist.boardings.median()))\n    q[\'seasonal7\'] = q.seasonal7.fillna(q.baseline)\n    q[\'weekday_mean\'] = q.board_mean_56.fillna(q.baseline)\n    q = q.sort_values(\'_order\').drop(columns=[\'_order\',\'date\']).reset_index(drop=True)\n    return q.astype(\'float32\')\n\ndef training_matrix(train, metadata, cutoff, horizon=123, origin_stride=28, max_rows=220000, seed=2026):\n    """Whole label windows end by cutoff; last training origin is horizon days earlier."""\n    cutoff = pd.Timestamp(cutoff)\n    start = train.date.min() + pd.Timedelta(days=27)\n    end = cutoff - pd.Timedelta(days=horizon)\n    origins = pd.date_range(start,end,freq=f\'{origin_stride}D\')\n    if len(origins) == 0: raise ValueError(\'Insufficient history for full-horizon training\')\n    # Include the exact last permissible anchor so the newest observed labels\n    # are represented even when the 28-day grid misses the cutoff boundary.\n    if origins[-1] != end: origins = origins.append(pd.DatetimeIndex([end]))\n    rng = np.random.default_rng(seed)\n    per_origin = max(1,max_rows//len(origins))\n    xs, ys, audit = [], [], []\n    for origin in origins:\n        labels = train.loc[(train.date > origin)&(train.date <= origin + pd.Timedelta(days=horizon))].copy()\n        if len(labels)>per_origin:\n            labels = labels.iloc[np.sort(rng.choice(len(labels),per_origin,replace=False))]\n        x = make_features(train,labels,metadata,origin)\n        xs.append(x); ys.append(labels.boardings.to_numpy(dtype=\'float32\'))\n        audit.append({\'origin\':str(origin.date()),\'feature_max_date\':str(origin.date()),\'label_end\':str((origin+pd.Timedelta(days=horizon)).date()),\'rows\':len(labels)})\n    return pd.concat(xs,ignore_index=True),np.concatenate(ys),audit\n',
    'src/holiday_calendar.py': '"""South Korean known-in-advance holiday covariates, version 2026-06-30.\n\nOnly calendar facts/public declarations issued by the competition cutoff are\nencoded. No ridership, weather realization or event outcome is used. Sources,\npublication dates and reuse notes: reports/holiday_sources.md.\n\n`is_holiday` means a named national holiday, substitute or election holiday;\nordinary Sundays are represented separately by `is_weekend` / `is_rest_day`.\nPass the forecast origin as `as_of` for historical CV. This hides a temporary\nholiday/new designation until the cited public announcement was available.\nNo feature reveals that a not-yet-announced designation will occur.\n"""\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\nfrom functools import lru_cache\nfrom typing import Iterable\n\nimport numpy as np\nimport pandas as pd\n\nKNOWLEDGE_CUTOFF = "2026-06-30"\nCOVERAGE_START = "2025-01-01"\nCOVERAGE_END = "2026-12-31"\n\nHOLIDAY_TYPES = {\n    "none": 0, "new_year": 1, "seollal": 2, "independence": 3,\n    "children": 4, "buddha": 5, "children_and_buddha": 6, "memorial": 7,\n    "liberation": 8, "chuseok": 9, "national_foundation": 10,\n    "hangul": 11, "christmas": 12, "temporary": 13,\n    "presidential_election": 14, "local_election": 15,\n    "labor": 16, "constitution": 17,\n}\n\n\n@dataclass(frozen=True)\nclass Holiday:\n    name: str\n    kind: str\n    known_since: str\n    source: str\n    substitute: bool = False\n\n\ndef _h(name, kind, year, substitute=False):\n    # Conservative availability of the final annual calendar announcement.\n    known = "2024-06-20" if year == 2025 else "2025-06-30"\n    return Holiday(name, kind, known, f"annual_{year}", substitute)\n\n\nHOLIDAYS = {\n    "2025-01-01": _h("신정", "new_year", 2025),\n    "2025-01-27": Holiday("임시공휴일", "temporary", "2025-01-14", "temporary_20250127"),\n    "2025-01-28": _h("설날 전날", "seollal", 2025),\n    "2025-01-29": _h("설날", "seollal", 2025),\n    "2025-01-30": _h("설날 다음날", "seollal", 2025),\n    "2025-03-01": _h("삼일절", "independence", 2025),\n    "2025-03-03": _h("삼일절 대체공휴일", "independence", 2025, True),\n    "2025-05-05": _h("어린이날·부처님오신날", "children_and_buddha", 2025),\n    "2025-05-06": _h("부처님오신날 대체공휴일", "buddha", 2025, True),\n    "2025-06-03": Holiday("제21대 대통령선거", "presidential_election", "2025-04-08", "election_20250603"),\n    "2025-06-06": _h("현충일", "memorial", 2025),\n    "2025-08-15": _h("광복절", "liberation", 2025),\n    "2025-10-03": _h("개천절", "national_foundation", 2025),\n    "2025-10-05": _h("추석 전날", "chuseok", 2025),\n    "2025-10-06": _h("추석", "chuseok", 2025),\n    "2025-10-07": _h("추석 다음날", "chuseok", 2025),\n    "2025-10-08": _h("추석 대체공휴일", "chuseok", 2025, True),\n    "2025-10-09": _h("한글날", "hangul", 2025),\n    "2025-12-25": _h("기독탄신일", "christmas", 2025),\n    "2026-01-01": _h("신정", "new_year", 2026),\n    "2026-02-16": _h("설날 전날", "seollal", 2026),\n    "2026-02-17": _h("설날", "seollal", 2026),\n    "2026-02-18": _h("설날 다음날", "seollal", 2026),\n    "2026-03-01": _h("삼일절", "independence", 2026),\n    "2026-03-02": _h("삼일절 대체공휴일", "independence", 2026, True),\n    "2026-05-01": Holiday("노동절", "labor", "2026-04-06", "labor_2026"),\n    "2026-05-05": _h("어린이날", "children", 2026),\n    "2026-05-24": _h("부처님오신날", "buddha", 2026),\n    "2026-05-25": _h("부처님오신날 대체공휴일", "buddha", 2026, True),\n    "2026-06-03": _h("제9회 전국동시지방선거", "local_election", 2026),\n    "2026-06-06": _h("현충일", "memorial", 2026),\n    "2026-07-17": Holiday("제헌절", "constitution", "2026-02-03", "constitution_2026"),\n    "2026-08-15": _h("광복절", "liberation", 2026),\n    "2026-08-17": _h("광복절 대체공휴일", "liberation", 2026, True),\n    "2026-09-24": _h("추석 전날", "chuseok", 2026),\n    "2026-09-25": _h("추석", "chuseok", 2026),\n    "2026-09-26": _h("추석 다음날", "chuseok", 2026),\n    # 2026-09-28 is NOT a substitute holiday. Chuseok on Saturday alone does\n    # not add a substitute day; the official calendar says Sep 24-27 break.\n    "2026-10-03": _h("개천절", "national_foundation", 2026),\n    "2026-10-05": _h("개천절 대체공휴일", "national_foundation", 2026, True),\n    "2026-10-09": _h("한글날", "hangul", 2026),\n    "2026-12-25": _h("기독탄신일", "christmas", 2026),\n}\n\n\ndef holiday_records() -> pd.DataFrame:\n    """All encoded facts and provenance; not a model feature table."""\n    return pd.DataFrame([\n        {"date": date, "name": h.name, "kind": h.kind,\n         "holiday_type": HOLIDAY_TYPES[h.kind], "known_since": h.known_since,\n         "source_id": h.source, "is_substitute": int(h.substitute)}\n        for date, h in sorted(HOLIDAYS.items())\n    ])\n\n\n@lru_cache(maxsize=128)\ndef _known_records(as_of: str):\n    return {pd.Timestamp(d): h for d, h in HOLIDAYS.items() if h.known_since <= as_of}\n\n\ndef calendar_features(dates: Iterable, as_of=KNOWLEDGE_CUTOFF) -> pd.DataFrame:\n    """Return numeric int8 features in the input row order.\n\n    A Series input preserves its index. Other iterables receive a RangeIndex.\n    `as_of` is one forecast origin, not a vector; it must be <= 2026-06-30.\n    Scope is 2025-2026. Reject invalid/missing/out-of-range dates rather than\n    silently labeling an unsupported year\'s holidays as ordinary days.\n    """\n    origin = pd.Timestamp(as_of).normalize()\n    if pd.isna(origin) or origin > pd.Timestamp(KNOWLEDGE_CUTOFF):\n        raise ValueError(f"as_of must be a valid date <= {KNOWLEDGE_CUTOFF}")\n    index = dates.index if isinstance(dates, pd.Series) else None\n    d = pd.DatetimeIndex(pd.to_datetime(dates)).normalize()\n    if d.isna().any():\n        raise ValueError("calendar dates must not contain NaT")\n    if len(d) and (d.min() < pd.Timestamp(COVERAGE_START) or d.max() > pd.Timestamp(COVERAGE_END)):\n        raise ValueError(f"calendar only covers {COVERAGE_START} through {COVERAGE_END}")\n    known = _known_records(origin.strftime("%Y-%m-%d"))\n    # Unique-date lookup makes repeated station/hour rows inexpensive.\n    unique = d.unique()\n    lookup = {}\n    for day in unique:\n        h = known.get(day)\n        before = day - pd.Timedelta(days=1)\n        after = day + pd.Timedelta(days=1)\n        weekend = day.dayofweek >= 5\n        rest = weekend or h is not None\n        prev_rest = before.dayofweek >= 5 or before in known\n        next_rest = after.dayofweek >= 5 or after in known\n        kind = h.kind if h else "none"\n        lookup[day] = (\n            int(h is not None), HOLIDAY_TYPES[kind], int(h.substitute if h else False),\n            int(kind in {"temporary", "presidential_election"}),\n            int(kind in {"presidential_election", "local_election"}),\n            int(kind in {"seollal", "buddha", "children_and_buddha", "chuseok"}),\n            int(day.month == 5 and day.day == 1), int(weekend), int(rest),\n            int(after in known), int(before in known),\n            int(not rest and prev_rest and next_rest),\n        )\n    columns = [\n        "is_holiday", "holiday_type", "is_substitute_holiday", "is_temporary_holiday",\n        "is_election_day", "is_lunar_holiday", "is_labor_day", "calendar_is_weekend",\n        "is_rest_day", "is_holiday_eve", "is_after_holiday", "is_bridge_day",\n    ]\n    if len(d):\n        values = np.asarray([lookup[x] for x in d], dtype=np.int8)\n    else:\n        values = np.empty((0, len(columns)), dtype=np.int8)\n    return pd.DataFrame(values, columns=columns, index=index)\n\n\nif __name__ == "__main__":\n    print(holiday_records().to_string(index=False))\n',
    'src/metrics.py': '"""Forecast metrics and calendar-aware diagnostics for SINCON.\n\nRMSLE is the official metric in the user-provided competition rules. ``smape`` uses\n200 * |prediction - target| / (|target| + |prediction|), with 0/0 := 0.\nMASE scales are calculated only from the supplied training history using\nexact seven-calendar-day matches within station/hour, never adjacent rows.\n"""\nfrom __future__ import annotations\n\nfrom typing import Sequence\n\nimport numpy as np\nimport pandas as pd\n\n\nSUPPORTED_METRICS = ("mae", "rmse", "smape", "rmsle")\nGROUP_COLS = ("station_id", "hour")\n\n\ndef _arrays(y_true, y_pred):\n    y = np.asarray(y_true, dtype=np.float64)\n    p = np.asarray(y_pred, dtype=np.float64)\n    if y.ndim != 1 or p.ndim != 1 or y.shape != p.shape or y.size == 0:\n        raise ValueError("Targets and predictions must be nonempty 1-D arrays of equal length.")\n    if not np.isfinite(y).all() or not np.isfinite(p).all():\n        raise ValueError("Targets and predictions must contain only finite numbers.")\n    return y, p\n\n\ndef regression_metrics(y_true, y_pred) -> dict[str, float]:\n    """Return unweighted row-level metrics; undefined values are NaN.\n\n    RMSLE is undefined for negative values. No implicit clipping is performed.\n    ``rmse_mae_ratio`` is undefined for perfect predictions (MAE == 0).\n    MBE is prediction minus truth, so positive means overprediction.\n    """\n    y, p = _arrays(y_true, y_pred)\n    err = p - y\n    abs_err = np.abs(err)\n    mae = float(abs_err.mean())\n    rmse = float(np.sqrt(np.mean(err * err)))\n    denom = np.abs(y) + np.abs(p)\n    smape_terms = np.divide(200.0 * abs_err, denom, out=np.zeros_like(err), where=denom > 0)\n    rmsle = float(np.sqrt(np.mean((np.log1p(p) - np.log1p(y)) ** 2))) if (y >= 0).all() and (p >= 0).all() else float("nan")\n    return {\n        "mae": mae,\n        "rmse": rmse,\n        "smape": float(smape_terms.mean()),\n        "rmsle": rmsle,\n        "mbe": float(err.mean()),\n        "rmse_mae_ratio": rmse / mae if mae > 0 else float("nan"),\n    }\n\n\ndef score(y_true, y_pred, metric: str = "rmsle") -> float:\n    """Select an objective explicitly; default is official RMSLE, all minimize."""\n    metric = metric.lower()\n    if metric not in SUPPORTED_METRICS:\n        raise ValueError(f"Unknown metric {metric!r}; choose {SUPPORTED_METRICS}.")\n    # Ensemble weight search calls this repeatedly: compute only its objective.\n    y, p = _arrays(y_true, y_pred)\n    if metric == "mae":\n        result = float(np.abs(p - y).mean())\n    elif metric == "rmse":\n        result = float(np.sqrt(np.mean((p - y) ** 2)))\n    elif metric == "smape":\n        denominator = np.abs(y) + np.abs(p)\n        result = float(np.divide(200.0 * np.abs(p - y), denominator, out=np.zeros_like(y), where=denominator > 0).mean())\n    else:\n        result = float(np.sqrt(np.mean((np.log1p(p) - np.log1p(y)) ** 2))) if (y >= 0).all() and (p >= 0).all() else float("nan")\n    if not np.isfinite(result):\n        raise ValueError(f"{metric} is undefined for these targets/predictions.")\n    return result\n\n\ndef _calendar_frame(frame: pd.DataFrame, group_cols: Sequence[str], target: str) -> pd.DataFrame:\n    needed = [*group_cols, "date", target]\n    missing = set(needed) - set(frame.columns)\n    if missing:\n        raise ValueError(f"Missing columns: {sorted(missing)}")\n    result = frame[needed].copy()\n    result["date"] = pd.to_datetime(result["date"], errors="raise").dt.normalize()\n    if result[needed].isna().any().any():\n        raise ValueError("Calendar keys and target cannot contain missing values.")\n    if result.duplicated([*group_cols, "date"]).any():\n        raise ValueError("Duplicate station/hour/date keys are not valid for calendar diagnostics.")\n    values = pd.to_numeric(result[target], errors="raise").to_numpy(dtype=float)\n    if not np.isfinite(values).all():\n        raise ValueError("Target values must be finite.")\n    result[target] = values\n    return result\n\n\ndef seasonal_scales(\n    train: pd.DataFrame,\n    seasonality_days: int = 7,\n    target: str = "boardings",\n    group_cols: Sequence[str] = GROUP_COLS,\n) -> pd.DataFrame:\n    """Training-only MASE denominator and exact seasonal pair counts per series.\n\n    Zero denominators are retained as zero so the caller can distinguish a\n    constant series from a series with no historical seasonal pairs (NaN).\n    """\n    if not isinstance(seasonality_days, int) or seasonality_days <= 0:\n        raise ValueError("seasonality_days must be a positive integer.")\n    t = _calendar_frame(train, group_cols, target)\n    previous = t.rename(columns={target: "seasonal_value"})\n    previous["date"] += pd.Timedelta(days=seasonality_days)\n    paired = t.merge(previous, on=[*group_cols, "date"], how="left", validate="one_to_one")\n    paired["seasonal_abs_difference"] = (paired[target] - paired["seasonal_value"]).abs()\n    return paired.groupby(list(group_cols), observed=True, dropna=False).agg(\n        mase_scale=("seasonal_abs_difference", "mean"),\n        seasonal_pairs=("seasonal_abs_difference", "count"),\n    ).reset_index()\n\n\ndef residual_autocorrelation(\n    predictions: pd.DataFrame,\n    lags: Sequence[int] = (1, 7, 28),\n    group_cols: Sequence[str] = GROUP_COLS,\n) -> pd.DataFrame:\n    """Pearson residual correlation at exact calendar-day lags, per series.\n\n    Input needs date/group columns and residual=prediction-truth. This is a\n    pairwise calendar-lag correlation, not an ACF across concatenated stations.\n    Fewer than three matched pairs or zero variance produces NaN.\n    """\n    t = _calendar_frame(predictions, group_cols, "residual")\n    records = []\n    for lag in lags:\n        if not isinstance(lag, int) or lag <= 0:\n            raise ValueError("ACF lags must be positive integer calendar days.")\n        previous = t.rename(columns={"residual": "residual_lag"})\n        previous["date"] += pd.Timedelta(days=lag)\n        paired = t.merge(previous, on=[*group_cols, "date"], how="inner", validate="one_to_one")\n        values_by_group = {}\n        for keys, frame in paired.groupby(list(group_cols), observed=True, dropna=False):\n            keys = keys if isinstance(keys, tuple) else (keys,)\n            left = frame["residual"].to_numpy()\n            right = frame["residual_lag"].to_numpy()\n            corr = float(np.corrcoef(left, right)[0, 1]) if len(frame) >= 3 and left.std() > 0 and right.std() > 0 else float("nan")\n            values_by_group[keys] = (len(frame), corr)\n        for keys in t[list(group_cols)].drop_duplicates().itertuples(index=False, name=None):\n            n, corr = values_by_group.get(keys, (0, float("nan")))\n            records.append({**dict(zip(group_cols, keys)), "lag_days": lag, "pairs": n, "acf": corr})\n    return pd.DataFrame.from_records(records)\n\n\ndef forecast_diagnostics(\n    train: pd.DataFrame,\n    valid: pd.DataFrame,\n    predictions,\n    origin=None,\n    horizon_col: str = "horizon",\n    target: str = "boardings",\n    group_cols: Sequence[str] = GROUP_COLS,\n    seasonality_days: int = 7,\n) -> dict:\n    """Return summary and DataFrames for one fixed-origin forecast fold.\n\n    ``train`` must end no later than origin; all validation dates must follow\n    origin. Predictions must have the same row order as ``valid``. The summary\n    contains two MASE aggregations: mean per-series MASE (macro), and the mean\n    per-row absolute error divided by that row\'s training-series scale. Both\n    exclude unavailable/zero scales and report their coverage explicitly.\n    Per-step metrics use calendar-day horizon, not row position or hour number.\n    """\n    t = _calendar_frame(train, group_cols, target)\n    v = _calendar_frame(valid, group_cols, target)\n    if t.empty or v.empty:\n        raise ValueError("Training and validation must both be nonempty.")\n    cutoff = pd.Timestamp(origin).normalize() if origin is not None else t["date"].max()\n    if t["date"].max() > cutoff:\n        raise ValueError("Training includes observations after forecast origin (leakage).")\n    if v["date"].min() <= cutoff:\n        raise ValueError("Validation must lie strictly after forecast origin.")\n    y, p = _arrays(v[target], predictions)\n    v["prediction"] = p\n    v["residual"] = p - y\n    v["absolute_error"] = np.abs(p - y)\n    actual_horizon = (v["date"] - cutoff).dt.days.to_numpy()\n    if horizon_col in valid:\n        supplied_horizon = pd.to_numeric(valid[horizon_col], errors="raise").to_numpy()\n        if not np.array_equal(supplied_horizon, actual_horizon):\n            raise ValueError("Supplied horizon does not equal date minus forecast origin.")\n    v["horizon"] = actual_horizon\n    scales = seasonal_scales(t, seasonality_days, target, group_cols)\n    v = v.merge(scales, on=list(group_cols), how="left", sort=False, validate="many_to_one")\n    usable_scale = v["mase_scale"].where(v["mase_scale"] > 0)\n    v["scaled_absolute_error"] = v["absolute_error"] / usable_scale\n    entity_rows = []\n    for keys, frame in v.groupby(list(group_cols), observed=True, dropna=False):\n        keys = keys if isinstance(keys, tuple) else (keys,)\n        stats = regression_metrics(frame[target], frame["prediction"])\n        entity_rows.append({\n            **dict(zip(group_cols, keys)),\n            "rows": len(frame),\n            **stats,\n            "mase_scale": frame["mase_scale"].iloc[0],\n            "seasonal_pairs": frame["seasonal_pairs"].iloc[0],\n            "mase": frame["scaled_absolute_error"].mean(),\n        })\n    per_entity = pd.DataFrame.from_records(entity_rows)\n    per_step = pd.DataFrame.from_records([\n        {"horizon": int(h), "rows": len(frame), **regression_metrics(frame[target], frame["prediction"]), "mase": frame["scaled_absolute_error"].mean()}\n        for h, frame in v.groupby("horizon", sort=True)\n    ])\n    acf = residual_autocorrelation(v, group_cols=group_cols)\n    summary = {\n        **regression_metrics(y, p),\n        "mase_macro": float(per_entity["mase"].mean()),\n        "mase_row_weighted": float(v["scaled_absolute_error"].mean()),\n        "mase_series_coverage": float(per_entity["mase"].notna().mean()),\n        "mase_row_coverage": float(v["scaled_absolute_error"].notna().mean()),\n        "mase_undefined_series": int(per_entity["mase"].isna().sum()),\n        "mase_seasonality_days": seasonality_days,\n        "origin": str(cutoff.date()),\n        "forecast_rows": len(v),\n    }\n    for lag, frame in acf.groupby("lag_days"):\n        finite = frame["acf"].dropna()\n        summary[f"acf_{lag}d_macro"] = float(finite.mean())\n        summary[f"acf_{lag}d_abs_gt_0_1_fraction"] = float((finite.abs() > 0.1).mean()) if len(finite) else float("nan")\n        summary[f"acf_{lag}d_defined_series"] = int(len(finite))\n    return {"summary": summary, "per_entity": per_entity, "residual_acf": acf, "per_step": per_step}\n',
    'src/models.py': '"""Scratch-trained CPU tree baselines and a small tabular neural baseline."""\nimport numpy as np\nimport sys\n\nclass NeuralRegressor:\n    """MLP with training-only scaling and missing masks; not TabM/RevIN."""\n    def __init__(self,seed=2026,threads=4,device=\'cpu\',epochs=10):\n        self.seed,self.threads,self.device,self.epochs=seed,threads,device,epochs\n\n    def fit(self,x,y):\n        import torch\n        # Apple OpenMP runtimes used by tree wheels can deadlock with Torch\'s\n        # indexed tensor kernels. One Torch CPU thread avoids this interaction.\n        torch.manual_seed(self.seed); torch.set_num_threads(1 if sys.platform==\'darwin\' else self.threads)\n        if self.device.startswith(\'cuda\') and not torch.cuda.is_available():\n            raise RuntimeError(\'CUDA requested but unavailable\')\n        self.columns=[c for c in x.columns if c in [\'station_id\',\'hour\',\'dow\',\'canonical_line\',\'horizon\',\'month\',\'date_index\',\'weekend\',\'hour_sin\',\'hour_cos\',\'dow_sin\',\'dow_cos\',\'day_of_year_sin\',\'day_of_year_cos\',\'history_days\',\'cold_start\',\'origin_lag_1\',\'origin_lag_7\',\'origin_lag_28\',\'seasonal7\',\'seasonal7_alight\',\'recent_mean\',\'recent_std\',\'recent_alight\',\'baseline\',\'is_holiday\',\'holiday_type\',\'is_holiday_eve\',\'is_after_holiday\']]\n        a=x[self.columns].to_numpy(dtype=\'float32\')\n        self.fill=np.nanmedian(a,axis=0)\n        self.fill=np.nan_to_num(self.fill,nan=0).astype(\'float32\')\n        filled=np.where(np.isnan(a),self.fill,a)\n        self.mean=filled.mean(axis=0); self.std=filled.std(axis=0).clip(min=1)\n        z=self.transform(x)\n        # y is log1p(count). Squared error is exactly the square of RMSLE.\n        target=y-np.log1p(x.baseline.to_numpy())\n        self.net=torch.nn.Sequential(torch.nn.Linear(z.shape[1],96),torch.nn.ReLU(),torch.nn.Linear(96,64),torch.nn.ReLU(),torch.nn.Linear(64,1)).to(self.device)\n        optimizer=torch.optim.AdamW(self.net.parameters(),lr=0.001,weight_decay=0.001)\n        use_amp=self.device.startswith(\'cuda\')\n        scaler=torch.amp.GradScaler(\'cuda\',enabled=use_amp)\n        rng=np.random.default_rng(self.seed)\n        n=min(len(z),160000)\n        idx=rng.choice(len(z),n,replace=False)\n        xt=torch.from_numpy(z[idx]); yt=torch.tensor(target[idx],dtype=torch.float32)\n        self.losses=[]\n        self.net.train()\n        for epoch in range(self.epochs):\n            perm=torch.randperm(n); losses=[]\n            for ids in perm.split(2048):\n                xb=xt[ids].to(self.device); yb=yt[ids].to(self.device)\n                optimizer.zero_grad(set_to_none=True)\n                with torch.autocast(device_type=\'cuda\',dtype=torch.float16,enabled=use_amp):\n                    pred=self.net(xb).squeeze(1);loss=torch.nn.functional.mse_loss(pred,yb)\n                if not torch.isfinite(loss):raise FloatingPointError(\'Nonfinite neural loss\')\n                scaler.scale(loss).backward()\n                scaler.unscale_(optimizer)\n                torch.nn.utils.clip_grad_norm_(self.net.parameters(),1.0,error_if_nonfinite=True)\n                scaler.step(optimizer);scaler.update()\n                losses.append(float(loss.detach().cpu()))\n            self.losses.append(float(np.sqrt(np.mean(losses))))\n        self.net.to(\'cpu\');self.net.eval()\n        return self\n\n    def transform(self,x):\n        a=x[self.columns].to_numpy(dtype=\'float32\')\n        mask=np.isnan(a)\n        z=(np.where(mask,self.fill,a)-self.mean)/self.std\n        return np.concatenate([z,mask.astype(\'float32\')],axis=1).astype(\'float32\')\n\n    def predict(self,x):\n        import torch\n        z=self.transform(x);parts=[]\n        self.net.eval()\n        with torch.no_grad():\n            for start in range(0,len(x),8192):\n                parts.append(self.net(torch.from_numpy(z[start:start+8192])).squeeze(1).numpy())\n        return np.log1p(x.baseline.to_numpy())+np.concatenate(parts)\n\nclass Model:\n    def __init__(self,name,config):\n        self.name,self.config=name,config\n\n    def _frame(self,x):\n        if self.name==\'catboost\':\n            z=x.copy()\n            for col in [\'station_id\',\'canonical_line\',\'hour\',\'dow\']:z[col]=z[col].astype(\'int32\').astype(str)\n            return z\n        return x\n\n    def fit(self,x,y):\n        name=self.name;cfg=self.config\n        if name in [\'seasonal7\',\'weekday_mean\']:return self\n        if cfg[\'training_loss\']!=\'log_mse\':\n            raise ValueError(\'SINCON official RMSLE requires this implementation to train log_mse.\')\n        y=np.log1p(y)\n        if name.startswith(\'lightgbm\'):\n            from lightgbm import LGBMRegressor\n            large=name.endswith(\'large\')\n            self.estimator=LGBMRegressor(objective=\'regression\',n_estimators=220 if large else 150,learning_rate=0.055,num_leaves=63 if large else 31,min_child_samples=80,reg_lambda=3,colsample_bytree=0.9,subsample=0.9,subsample_freq=1,random_state=cfg[\'seed\'],n_jobs=cfg[\'threads\'],verbosity=-1,linear_tree=False)\n            self.estimator.fit(x,y,categorical_feature=[\'station_id\',\'canonical_line\',\'hour\',\'dow\'])\n        elif name==\'catboost\':\n            from catboost import CatBoostRegressor\n            self.estimator=CatBoostRegressor(iterations=220,depth=7,learning_rate=0.07,loss_function=\'RMSE\',l2_leaf_reg=5,random_seed=cfg[\'seed\'],thread_count=cfg[\'threads\'],verbose=False,allow_writing_files=False)\n            self.estimator.fit(self._frame(x),y,cat_features=[\'station_id\',\'canonical_line\',\'hour\',\'dow\'])\n        elif name==\'xgboost\':\n            from xgboost import XGBRegressor\n            self.estimator=XGBRegressor(objective=\'reg:squarederror\',n_estimators=180,max_depth=6,learning_rate=0.06,min_child_weight=30,reg_lambda=5,subsample=0.9,colsample_bytree=0.85,tree_method=\'hist\',n_jobs=cfg[\'threads\'],random_state=cfg[\'seed\'])\n            self.estimator.fit(x,y)\n        elif name==\'mlp\':\n            self.estimator=NeuralRegressor(cfg[\'seed\'],cfg[\'threads\'],cfg[\'device\']).fit(x,y)\n        else:raise ValueError(f\'Unknown model {name}\')\n        return self\n\n    def predict(self,x):\n        if self.name in [\'seasonal7\',\'weekday_mean\']:return x[self.name].to_numpy(dtype=\'float64\')\n        pred=np.expm1(np.maximum(0,self.estimator.predict(self._frame(x)))).astype(\'float64\')\n        if not np.isfinite(pred).all():raise ValueError(f\'{self.name}: nonfinite predictions\')\n        return pred\n',
    'src/pipeline.py': '"""Reproducible EDA -> fixed temporal CV -> fit -> verified CSV pipeline."""\nfrom __future__ import annotations\nimport argparse\nimport hashlib\nimport importlib.metadata\nimport json\nimport platform\nimport resource\nimport sys\nimport time\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\nfrom .features import load_data,make_features,training_matrix\nfrom .models import Model\nfrom .ensemble import fit_ensemble,predict_ensemble\nfrom .metrics import regression_metrics,forecast_diagnostics\nfrom .validate_submission import validate_submission\n\nROOT=Path(__file__).resolve().parents[1]\n\ndef clean(value):\n    if isinstance(value,dict):return {str(k):clean(v) for k,v in value.items()}\n    if isinstance(value,(list,tuple)):return [clean(v) for v in value]\n    if isinstance(value,(np.integer,)):return int(value)\n    if isinstance(value,(np.floating,float)):return float(value) if np.isfinite(value) else None\n    if isinstance(value,(np.bool_,)):return bool(value)\n    return value\n\ndef save_json(path,value):\n    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)\n    path.write_text(json.dumps(clean(value),ensure_ascii=False,indent=2,allow_nan=False)+\'\\n\',encoding=\'utf-8\')\n\ndef environment():\n    versions={}\n    for name in [\'numpy\',\'pandas\',\'polars\',\'scikit-learn\',\'scipy\',\'lightgbm\',\'catboost\',\'xgboost\',\'torch\',\'joblib\']:\n        try:versions[name]=importlib.metadata.version(name)\n        except importlib.metadata.PackageNotFoundError:versions[name]=None\n    return {\'python\':sys.version,\'platform\':platform.platform(),\'packages\':versions}\n\ndef diagnostics(history,valid,prediction,origin,path):\n    result=forecast_diagnostics(history.loc[history.date<=origin],valid,prediction,origin=origin)\n    for key in [\'per_entity\',\'residual_acf\',\'per_step\']:\n        result[key].to_csv(path/f\'{key}.csv\',index=False)\n    monthly=[]\n    v=valid.copy();v[\'prediction\']=prediction\n    for month,rows in v.groupby(v.date.dt.to_period(\'M\')):\n        monthly.append({\'month\':str(month),\'rows\':len(rows),**regression_metrics(rows.boardings,rows.prediction)})\n    pd.DataFrame(monthly).to_csv(path/\'per_month.csv\',index=False)\n    # Day-block bootstrap reflects the competition\'s date-level public/private grouping.\n    daily=v.assign(log_se=(np.log1p(v.prediction)-np.log1p(v.boardings))**2).groupby(\'date\').log_se.agg([\'sum\',\'size\'])\n    rng=np.random.default_rng(2026)\n    values=[]\n    for _ in range(500):\n        ix=rng.integers(0,len(daily),size=len(daily));sample=daily.iloc[ix]\n        values.append(float(np.sqrt(sample[\'sum\'].sum()/sample[\'size\'].sum())))\n    result[\'summary\'][\'day_bootstrap_rmsle_ci95\']=[float(np.quantile(values,.025)),float(np.quantile(values,.975))]\n    result[\'summary\'][\'day_bootstrap_note\']=\'500 date-block resamples; descriptive conditional interval, not a guarantee for future months.\'\n    save_json(path/\'summary.json\',result[\'summary\'])\n    return result[\'summary\']\n\ndef export_predictions(data,predictions,output,reference=None):\n    output.mkdir(parents=True,exist_ok=True)\n    full=data[\'test\'][[\'example_id\']].copy();full[\'boardings\']=predictions\n    full.to_csv(output/\'submission.csv\',index=False,float_format=\'%.8f\')\n    # Keep original stale template as evidence; this secondary file is diagnostic only.\n    subset=data[\'sample_submission\'][[\'example_id\']].merge(full,on=\'example_id\',how=\'left\',validate=\'one_to_one\')\n    subset.to_csv(output/\'template_diagnostic_DO_NOT_SUBMIT.csv\',index=False,float_format=\'%.8f\')\n    full_report=validate_submission(output/\'submission.csv\',data[\'test\'],data[\'sample_submission\'],scope=\'test\',reference_predictions=reference)\n    full_report[\'official_expected_rows\']=157194\n    full_report[\'official_scope_source\']=\'User-provided Overview/Data/Rules: 123 days and all test IDs; local sample is stale.\'\n    if len(full)!=157194:raise ValueError(\'Official SINCON row count must be 157194\')\n    save_json(output/\'submission_validation.json\',full_report)\n    rich=data[\'test\'].copy();rich[\'boardings\']=predictions\n    rich.to_csv(output/\'predictions_with_keys.csv\',index=False,float_format=\'%.8f\')\n    return full_report\n\ndef run(args):\n    import joblib\n    start=time.perf_counter()\n    cfg=json.loads(args.config.read_text())\n    if cfg[\'selection_metric\']!=\'rmsle\' or cfg[\'training_loss\']!=\'log_mse\':raise ValueError(\'Official metric/loss configuration mismatch\')\n    if len(cfg[\'fold_origins\'])!=len(cfg[\'fold_roles\']):raise ValueError(\'Fold role mismatch\')\n    if cfg[\'fold_roles\']!=[\'development\',\'development\',\'audit\']:raise ValueError(\'Expected two development folds followed by one evaluation fold\')\n    data=load_data(args.data_dir);train=data[\'train\'];meta=data[\'metaData\']\n    output=args.output;output.mkdir(parents=True,exist_ok=True)\n    save_json(output/\'config_resolved.json\',cfg)\n    save_json(output/\'environment.json\',environment())\n    fingerprints={p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in args.data_dir.glob(\'*.csv\')}\n    save_json(output/\'input_sha256.json\',fingerprints)\n    fold_predictions=[];fold_truth=[];fold_frames=[];metrics=[];audits=[];run_times=[]\n    ensemble_plan=None;development_pred=None\n    for fold,(date,role) in enumerate(zip(cfg[\'fold_origins\'],cfg[\'fold_roles\'])):\n        origin=pd.Timestamp(date);end=origin+pd.Timedelta(days=cfg[\'horizon_days\'])\n        if end>train.date.max():raise ValueError(\'Validation extends beyond observed training\')\n        valid=train.loc[(train.date>origin)&(train.date<=end)].copy().reset_index(drop=True)\n        print(f\'Fold {fold+1} ({role}) origin={date}; validation={len(valid):,}\',flush=True)\n        t=time.perf_counter()\n        x,y,anchor_audit=training_matrix(train,meta,origin,cfg[\'horizon_days\'],cfg[\'origin_stride_days\'],cfg[\'max_training_rows\'],cfg[\'seed\'])\n        xv=make_features(train,valid,meta,origin)\n        assert x.columns.equals(xv.columns)\n        fold_path=output/f\'fold_{fold+1}\';fold_path.mkdir(exist_ok=True)\n        save_json(fold_path/\'training_origins.json\',anchor_audit)\n        audits.append({\'fold\':fold+1,\'role\':role,\'origin\':date,\'validation_start\':str(valid.date.min().date()),\'validation_end\':str(valid.date.max().date()),\'validation_rows\':len(valid),\'training_sample_rows\':len(y),\'feature_count\':x.shape[1],\'training_matrix_mb\':float(x.memory_usage(deep=True).sum()/1e6),\'training_anchor_max\':anchor_audit[-1][\'origin\'],\'training_label_end_max\':max(a[\'label_end\'] for a in anchor_audit),\'full_horizon_embargo\':True})\n        print(f\'  Features={x.shape}, created in {time.perf_counter()-t:.1f}s\',flush=True)\n        preds={}\n        for name in cfg[\'models\']:\n            t=time.perf_counter();model=Model(name,cfg).fit(x,y)\n            preds[name]=model.predict(xv)\n            elapsed=time.perf_counter()-t\n            if not np.isfinite(preds[name]).all():raise ValueError(f\'{name}: nonfinite output\')\n            row={\'fold\':fold+1,\'role\':role,\'model\':name,\'seconds\':elapsed,**regression_metrics(valid.boardings,preds[name])}\n            metrics.append(row);run_times.append({\'phase\':\'cv\',\'fold\':fold+1,\'model\':name,\'seconds\':elapsed})\n            print(f\'  {name:18s} RMSLE={row["rmsle"]:.6f} MAE={row["mae"]:.3f} ({elapsed:.1f}s)\',flush=True)\n            del model\n        frame=valid[[\'date\',\'station_id\',\'hour\',\'boardings\']].copy()\n        frame[\'horizon\']=(valid.date-origin).dt.days\n        for name,pred in preds.items():frame[name]=pred\n        fold_predictions.append(preds);fold_truth.append(valid.boardings.to_numpy());fold_frames.append(frame)\n        if role==\'development\' and cfg[\'fold_roles\'][fold+1]==\'audit\':\n            dev_y=np.concatenate(fold_truth)\n            dev_preds={name:np.concatenate([p[name] for p in fold_predictions]) for name in cfg[\'models\']}\n            ensemble_plan,corr=fit_ensemble(dev_y,dev_preds,cfg[\'selection_metric\'],cfg[\'correlation_cutoff\'],cfg.get(\'ensemble_policy\'))\n            ensemble_plan[\'fit_folds\']=[i+1 for i in range(fold+1)]\n            ensemble_plan[\'audit_fold_used_for_tuning\']=cfg.get(\'evaluation_context\',{}).get(\'policy_selection_has_seen_audit\',False)\n            ensemble_plan[\'audit_fold_used_for_weight_fit\']=False\n            ensemble_plan[\'evaluation_context\']=cfg.get(\'evaluation_context\',{\'status\':\'original_audit\'})\n            save_json(output/\'ensemble_plan.json\',ensemble_plan)\n            corr.to_csv(output/\'development_log_residual_correlations.csv\')\n            development_pred=predict_ensemble(dev_preds,ensemble_plan)\n            print(\'  Development-only ensemble frozen:\',ensemble_plan[\'weights\'],flush=True)\n        if role==\'audit\':\n            if ensemble_plan is None:raise ValueError(\'No development-only ensemble plan\')\n            pred=predict_ensemble(preds,ensemble_plan);frame[\'ensemble\']=pred\n            summary=diagnostics(train,valid,pred,origin,fold_path)\n            metrics.append({\'fold\':fold+1,\'role\':role,\'model\':\'ensemble\',**summary})\n            print(f\'  AUDIT ensemble RMSLE={summary["rmsle"]:.6f}\',flush=True)\n            save_json(output/\'audit_summary.json\',summary)\n        frame.to_csv(fold_path/\'oof.csv.gz\',index=False,compression=\'gzip\')\n        pd.DataFrame(metrics).to_csv(output/\'cv_scores.csv\',index=False)\n        save_json(output/\'cv_folds.json\',audits)\n        del x,y,xv\n    for fold in [0,1]:\n        frame=fold_frames[fold]\n        pred=predict_ensemble(fold_predictions[fold],ensemble_plan)\n        path=output/f\'fold_{fold+1}\'\n        diagnostics(train,frame,pred,pd.Timestamp(cfg[\'fold_origins\'][fold]),path)\n        frame[\'ensemble\']=pred\n        frame.to_csv(path/\'oof.csv.gz\',index=False,compression=\'gzip\')\n    print(\'Fitting selected nonzero-weight models on all observed data...\',flush=True)\n    cutoff=train.date.max()\n    x,y,final_audit=training_matrix(train,meta,cutoff,cfg[\'horizon_days\'],cfg[\'origin_stride_days\'],cfg[\'max_training_rows\'],cfg[\'seed\'])\n    xt=make_features(train,data[\'test\'],meta,cutoff)\n    save_json(output/\'final_training_origins.json\',final_audit)\n    models={};test_preds={}\n    for name,weight in ensemble_plan[\'weights\'].items():\n        if weight<=0:continue\n        t=time.perf_counter();model=Model(name,cfg).fit(x,y)\n        test_preds[name]=model.predict(xt);models[name]=model\n        elapsed=time.perf_counter()-t;run_times.append({\'phase\':\'final_fit_predict\',\'model\':name,\'seconds\':elapsed})\n        print(f\'  Final {name}: weight={weight:.5f}, {elapsed:.1f}s\',flush=True)\n    predictions=predict_ensemble(test_preds,ensemble_plan)\n    bundle={\'config\':cfg,\'ensemble\':ensemble_plan,\'models\':models,\'features\':x.columns.tolist(),\'training_cutoff\':str(cutoff.date()),\'input_sha256\':fingerprints}\n    joblib.dump(bundle,output/\'model_bundle.joblib\',compress=3)\n    export_predictions(data,predictions,output,fold_frames[-1][\'ensemble\'].to_numpy())\n    # Record a strong single-model alternative without using holdout to choose it.\n    best_name=min(ensemble_plan[\'individual_scores\'],key=ensemble_plan[\'individual_scores\'].get)\n    if best_name in test_preds:\n        alternative=data[\'test\'][[\'example_id\']].copy();alternative[\'boardings\']=test_preds[best_name]\n        alternative.to_csv(output/\'submission_best_single.csv\',index=False,float_format=\'%.8f\')\n        validate_submission(output/\'submission_best_single.csv\',data[\'test\'],data[\'sample_submission\'],scope=\'test\')\n    primary_name=cfg.get(\'ensemble_policy\',{}).get(\'primary_model\')\n    if primary_name in test_preds:\n        alternative=data[\'test\'][[\'example_id\']].copy();alternative[\'boardings\']=test_preds[primary_name]\n        alternative.to_csv(output/\'submission_primary.csv\',index=False,float_format=\'%.8f\')\n        validate_submission(output/\'submission_primary.csv\',data[\'test\'],data[\'sample_submission\'],scope=\'test\')\n    peak=resource.getrusage(resource.RUSAGE_SELF).ru_maxrss\n    peak_mb=peak/(1024*1024) if sys.platform==\'darwin\' else peak/1024\n    save_json(output/\'run_summary.json\',{\'status\':\'completed\',\'official_metric\':\'rmsle\',\'elapsed_seconds\':time.perf_counter()-start,\'peak_rss_mb\':peak_mb,\'feature_count\':x.shape[1],\'models_evaluated\':cfg[\'models\'],\'models_deployed\':list(models),\'timings\':run_times,\'external_data\':[\'preannounced Korean calendar facts\'],\'pretrained_weights\':False,\'submission_rows\':len(predictions),\'submission_source\':\'all test IDs\',\'audit_rmsle\':metrics[-1][\'rmsle\']})\n    print(f\'Completed: {output / "submission.csv"}; {len(predictions):,} rows\',flush=True)\n\ndef infer(args):\n    import joblib\n    data=load_data(args.data_dir)\n    bundle=joblib.load(args.model) # Load only this project\'s trusted locally generated artifact.\n    if str(data[\'train\'].date.max().date())!=bundle[\'training_cutoff\']:raise ValueError(\'Training cutoff mismatch\')\n    for filename,digest in bundle[\'input_sha256\'].items():\n        if hashlib.sha256((args.data_dir/filename).read_bytes()).hexdigest()!=digest:raise ValueError(f\'Input changed: {filename}\')\n    x=make_features(data[\'train\'],data[\'test\'],data[\'metaData\'],bundle[\'training_cutoff\'])\n    if x.columns.tolist()!=bundle[\'features\']:raise ValueError(\'Feature schema mismatch\')\n    preds={name:model.predict(x) for name,model in bundle[\'models\'].items()}\n    report=export_predictions(data,predict_ensemble(preds,bundle[\'ensemble\']),args.output)\n    print(json.dumps(clean(report),ensure_ascii=False,indent=2))\n\ndef main():\n    parser=argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'command\',choices=[\'train\',\'infer\'])\n    parser.add_argument(\'--data-dir\',type=Path,default=ROOT/\'data\'/\'raw\')\n    parser.add_argument(\'--config\',type=Path,default=ROOT/\'configs\'/\'default.json\')\n    parser.add_argument(\'--output\',type=Path,default=ROOT/\'outputs\'/\'three_tree\')\n    parser.add_argument(\'--model\',type=Path,default=ROOT/\'outputs\'/\'three_tree\'/\'model_bundle.joblib\')\n    args=parser.parse_args()\n    run(args) if args.command==\'train\' else infer(args)\n\nif __name__==\'__main__\':main()\n',
    'src/validate_submission.py': '"""Strict SINCON submission validation; never repairs or prunes input rows."""\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport sys\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\n\nEXPECTED_COLUMNS = ["example_id", "boardings"]\n\n\ndef _read(frame_or_path) -> pd.DataFrame:\n    if isinstance(frame_or_path, pd.DataFrame):\n        return frame_or_path.copy()\n    return pd.read_csv(frame_or_path, dtype={"example_id": "string"})\n\n\ndef _ids(frame: pd.DataFrame, name: str) -> pd.Series:\n    if "example_id" not in frame:\n        raise ValueError(f"{name}: example_id column is missing.")\n    ids = frame["example_id"]\n    if ids.isna().any() or ids.astype(str).str.strip().eq("").any():\n        raise ValueError(f"{name}: example_id contains missing/empty values.")\n    if not ids.map(lambda x: isinstance(x, str)).all():\n        raise ValueError(f"{name}: example_id must be strings; preserve original identifiers.")\n    if ids.duplicated().any():\n        raise ValueError(f"{name}: duplicate example_id values.")\n    return ids.astype(str).reset_index(drop=True)\n\n\ndef _distribution(values) -> dict:\n    v = np.asarray(values, dtype=float).reshape(-1)\n    if len(v) == 0 or not np.isfinite(v).all():\n        raise ValueError("Distribution input must contain finite predictions.")\n    return {\n        "count": len(v), "min": float(v.min()), "mean": float(v.mean()),\n        "p01": float(np.quantile(v, 0.01)), "p50": float(np.quantile(v, 0.5)),\n        "p99": float(np.quantile(v, 0.99)), "max": float(v.max()),\n        "zero_fraction": float((v == 0).mean()),\n    }\n\n\ndef validate_submission(submission, test, sample, scope: str = "test", reference_predictions=None) -> dict:\n    """Validate exact schema, scope-specific ID order, values, and scope mismatch.\n\n    ``test`` is the officially prescribed 123-day scope and expects every test\n    ID. ``template`` checks the stale 92-day sample only for diagnostics; passing\n    that check does not make its file ready for the official full-test scope.\n    The discrepancy is reported in either mode and never silently resolved.\n    """\n    if scope not in ("template", "test"):\n        raise ValueError("scope must be \'template\' or \'test\'.")\n    sub, tst, tmpl = (_read(x) for x in (submission, test, sample))\n    if list(sub.columns) != EXPECTED_COLUMNS:\n        raise ValueError(f"Submission columns/order must equal {EXPECTED_COLUMNS}, got {list(sub.columns)}.")\n    if list(tmpl.columns) != EXPECTED_COLUMNS:\n        raise ValueError(f"Sample columns/order must equal {EXPECTED_COLUMNS}.")\n    sub_ids, test_ids, sample_ids = _ids(sub, "submission"), _ids(tst, "test"), _ids(tmpl, "sample")\n    if len(sub) == 0:\n        raise ValueError("Submission cannot be empty.")\n    expected = sample_ids if scope == "template" else test_ids\n    if len(sub_ids) != len(expected):\n        raise ValueError(f"Submission has {len(sub_ids)} rows but {scope} scope requires {len(expected)}.")\n    if not np.array_equal(sub_ids.to_numpy(), expected.to_numpy()):\n        raise ValueError(f"Submission example_id values/order differ from {scope} scope.")\n    pred = pd.to_numeric(sub["boardings"], errors="raise").to_numpy(dtype=float)\n    if not np.isfinite(pred).all():\n        raise ValueError("Submission predictions contain NaN or infinite values.")\n    if (pred < 0).any():\n        raise ValueError("Submission predictions contain negative boardings.")\n    test_set, sample_set = set(test_ids), set(sample_ids)\n    missing_from_template = test_set - sample_set\n    unknown_template_ids = sample_set - test_set\n    if unknown_template_ids:\n        raise ValueError(f"Sample contains {len(unknown_template_ids)} IDs absent from test.")\n    warnings = []\n    scope_agrees = np.array_equal(test_ids.to_numpy(), sample_ids.to_numpy())\n    if not scope_agrees:\n        warnings.append(\n            f"Stale sample template: sample={len(sample_ids)} rows, test={len(test_ids)} rows; "\n            f"{len(missing_from_template)} test IDs are absent from the sample. "\n            "Official submission scope is the full test ID sequence. "\n            + ("Validated that full test scope." if scope == "test" else "Template scope is diagnostic only and is not the official submission scope.")\n        )\n    mismatch = {"template_rows": len(sample_ids), "test_rows": len(test_ids), "test_ids_absent_from_template": len(missing_from_template)}\n    if "date" in tst and missing_from_template:\n        dates = tst.loc[tst["example_id"].astype(str).isin(missing_from_template), "date"].astype(str)\n        mismatch.update({"missing_period_start": dates.min(), "missing_period_end": dates.max()})\n    result = {\n        "valid": True, "scope": scope, "rows": len(sub), "columns": list(sub.columns),\n        "official_scope": "test", "matches_official_scope": scope == "test",\n        "exact_id_order": True, "finite_nonnegative": True,\n        "template_matches_test": bool(scope_agrees), "scope_discrepancy": mismatch,\n        "warnings": warnings, "prediction_distribution": _distribution(pred),\n    }\n    if reference_predictions is not None:\n        ref = _distribution(reference_predictions)\n        result["reference_prediction_distribution"] = ref\n        result["distribution_comparison"] = {\n            "mean_ratio": result["prediction_distribution"]["mean"] / ref["mean"] if ref["mean"] != 0 else None,\n            "median_ratio": result["prediction_distribution"]["p50"] / ref["p50"] if ref["p50"] != 0 else None,\n            "note": "Diagnostic only: forecast dates/composition differ; ratios are not automatic rejection rules.",\n        }\n    return result\n\n\ndef main(argv=None) -> int:\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--submission", required=True, type=Path)\n    parser.add_argument("--test", required=True, type=Path)\n    parser.add_argument("--sample", required=True, type=Path)\n    parser.add_argument("--scope", choices=("template", "test"), default="test")\n    parser.add_argument("--reference", type=Path, help="Optional validation prediction CSV.")\n    parser.add_argument("--reference-column", default="prediction")\n    parser.add_argument("--report", type=Path, help="Optional JSON output path.")\n    args = parser.parse_args(argv)\n    try:\n        ref = None\n        if args.reference:\n            reference = pd.read_csv(args.reference)\n            ref = reference[args.reference_column].to_numpy()\n        result = validate_submission(args.submission, args.test, args.sample, args.scope, ref)\n    except (ValueError, KeyError, OSError) as error:\n        print(f"Submission validation failed: {error}", file=sys.stderr)\n        return 2\n    content = json.dumps(result, indent=2, ensure_ascii=False, allow_nan=False)\n    print(content)\n    if args.report:\n        args.report.parent.mkdir(parents=True, exist_ok=True)\n        args.report.write_text(content + "\\n", encoding="utf-8")\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n',
    'tests/test_ensemble.py': '"""OOF blend constraints, duplicate filtering, and MAE bias correction."""\nfrom pathlib import Path\nimport sys\nimport unittest\n\nimport numpy as np\n\nsys.path.insert(0, str(Path(__file__).resolve().parents[1]))\nfrom src.ensemble import fit_ensemble, predict_ensemble\nfrom src.metrics import score\n\n\nclass EnsembleTests(unittest.TestCase):\n    def test_primary_constraint_and_no_bias_match_boundary_optimum(self):\n        truth=np.expm1(np.array([2.,3.,4.,5.]))\n        predictions={\'large\':np.expm1(np.log1p(truth)+.4),\n                     \'cat\':np.expm1(np.log1p(truth)-.1),\n                     \'xgb\':np.expm1(np.log1p(truth)+.6)}\n        policy={\'method\':\'primary_constrained_log_mse\',\'primary_model\':\'large\',\n                \'primary_min_weight\':.5,\'bias_correction\':False}\n        plan,_=fit_ensemble(truth,predictions,policy=policy)\n        # Unconstrained optimum can cancel bias with large=.2, cat=.8.\n        # Requiring large>=.5 makes [.5,.5,0] the known constrained optimum.\n        np.testing.assert_allclose([plan[\'weights\'][n] for n in predictions],[.5,.5,0],atol=1e-8)\n        self.assertEqual(plan[\'bias\'],0.)\n        self.assertEqual(plan[\'excluded\'],{})\n        self.assertAlmostEqual(plan[\'development_score\'],.15,places=10)\n\n    def test_primary_blend_keeps_correlated_models_and_is_order_invariant(self):\n        truth=np.expm1(np.array([2.,3.,4.,5.]))\n        error=np.array([.1,-.2,.3,-.1])\n        predictions={\'large\':np.expm1(np.log1p(truth)+error),\n                     \'cat\':np.expm1(np.log1p(truth)+error*.9),\n                     \'xgb\':np.expm1(np.log1p(truth)-error*.7)}\n        policy={\'method\':\'primary_constrained_log_mse\',\'primary_model\':\'large\',\n                \'primary_min_weight\':.5,\'bias_correction\':False}\n        plan,corr=fit_ensemble(truth,predictions,policy=policy)\n        other,_=fit_ensemble(truth,dict(reversed(list(predictions.items()))),policy=policy)\n        self.assertGreater(corr.loc[\'large\',\'cat\'],.95)\n        self.assertEqual(set(plan[\'retained\']),set(predictions))\n        self.assertAlmostEqual(sum(plan[\'weights\'].values()),1.,places=12)\n        self.assertGreaterEqual(plan[\'weights\'][\'large\'],.5-1e-12)\n        np.testing.assert_allclose(predict_ensemble(predictions,plan),predict_ensemble(predictions,other),rtol=1e-8)\n\n    def test_primary_blend_rejects_invalid_inputs(self):\n        policy={\'method\':\'primary_constrained_log_mse\',\'primary_model\':\'large\',\'primary_min_weight\':.5}\n        with self.assertRaises(ValueError):fit_ensemble([1.,2.],{\'large\':np.array([1.,np.nan])},policy=policy)\n        with self.assertRaises(ValueError):fit_ensemble([1.,2.],{\'large\':np.array([-1.,2.])},policy=policy)\n        with self.assertRaises(ValueError):fit_ensemble([1.,2.],{\'cat\':np.array([1.,2.])},policy=policy)\n        with self.assertRaises(ValueError):fit_ensemble([1.,2.],{\'large\':np.array([1.,2.])},metric=\'mae\',policy=policy)\n\n    def test_convex_weights_and_duplicate_residual_filter(self):\n        truth = np.array([10., 15., 20., 25., 30., 35., 40., 45.])\n        error = np.array([1., -3., 4., -1., 2., -4., 3., -2.])\n        predictions = {"model_a": truth + error, "duplicate_a": truth + error, "model_b": truth - error * .8 + 0.4}\n        plan, correlation = fit_ensemble(truth, predictions, metric="mae")\n        self.assertTrue(all(weight >= 0 for weight in plan["weights"].values()))\n        self.assertAlmostEqual(sum(plan["weights"].values()), 1., places=12)\n        self.assertLessEqual(len(set(plan["retained"]) & {"model_a", "duplicate_a"}), 1)\n        for i, left in enumerate(plan["retained"]):\n            for right in plan["retained"][i + 1:]:\n                self.assertFalse(correlation.loc[left, right] > .95)\n        blended = predict_ensemble(predictions, plan)\n        self.assertTrue(np.isfinite(blended).all())\n        self.assertTrue((blended >= 0).all())\n        self.assertLessEqual(score(truth, blended, \'mae\'), min(score(truth, p, \'mae\') for p in predictions.values()))\n\n    def test_mae_bias_uses_median_and_improves_objective(self):\n        truth = np.array([10., 20., 30., 40., 50.])\n        # Residual mean is -0.2 while the median correction is +5.\n        predictions = {"only": truth + np.array([-5., -5., -5., -5., 19.])}\n        plan, _ = fit_ensemble(truth, predictions, metric="mae")\n        self.assertEqual(plan["bias"], 5.)\n        result = predict_ensemble(predictions, plan)\n        self.assertLess(score(truth, result, \'mae\'), score(truth, predictions["only"], \'mae\'))\n        self.assertAlmostEqual(plan["development_score"], score(truth, result, \'mae\'))\n\n    def test_rmse_does_not_apply_mae_median_correction(self):\n        truth = np.array([10., 20., 30.])\n        predictions = {"only": truth - 5}\n        plan, _ = fit_ensemble(truth, predictions, metric="rmse")\n        self.assertEqual(plan["bias"], 0.)\n\n    def test_rmsle_blends_and_calibrates_in_log1p_space(self):\n        truth_log = np.array([1., 2., 3., 4., 5.])\n        truth = np.expm1(truth_log)\n        residual_log = np.array([-.2, -.2, -.2, -.2, .6])\n        predictions = {"only": np.expm1(truth_log + residual_log)}\n        plan, _ = fit_ensemble(truth, predictions)\n        self.assertEqual(plan["metric"], "rmsle")\n        self.assertEqual(plan["blend_space"], "log1p")\n        self.assertEqual(plan["bias_space"], "log1p")\n        # RMSLE minimizes squared error in log space, so use mean, not median.\n        self.assertAlmostEqual(plan["bias"], .04, places=12)\n        expected = np.expm1(truth_log + residual_log + .04)\n        actual = predict_ensemble(predictions, plan)\n        np.testing.assert_allclose(actual, expected)\n        self.assertLess(score(truth, actual, "rmsle"), score(truth, predictions["only"], "rmsle"))\n\n    def test_predict_ensemble_uses_weighted_geometric_mean_of_one_plus_counts(self):\n        predictions = {"a": np.array([0., 3., 8.]), "b": np.array([3., 8., 15.])}\n        plan = {"weights": {"a": .25, "b": .75}, "bias": .1, "blend_space": "log1p"}\n        expected = np.exp(.1) * (1 + predictions["a"])**.25 * (1 + predictions["b"])**.75 - 1\n        np.testing.assert_allclose(predict_ensemble(predictions, plan), expected)\n        plan["bias"] = -10.\n        np.testing.assert_array_equal(predict_ensemble(predictions, plan), np.zeros(3))\n\n    def test_constant_residual_duplicates_are_removed_even_when_correlation_undefined(self):\n        truth = np.array([10., 20., 30.])\n        predictions = {"a": truth + 5, "duplicate": truth + 5}\n        plan, correlation = fit_ensemble(truth, predictions, metric="mae")\n        self.assertTrue(np.isnan(correlation.loc["a", "duplicate"]))\n        self.assertEqual(len(plan["retained"]), 1)\n        self.assertEqual(len(plan["excluded"]), 1)\n\n\nif __name__ == "__main__":\n    unittest.main()\n',
    'tests/test_features.py': '"""Leakage and horizon-boundary regression tests using independent toy data."""\nfrom pathlib import Path\nimport sys\nimport unittest\n\nimport numpy as np\nimport pandas as pd\nfrom pandas.testing import assert_frame_equal\n\nsys.path.insert(0, str(Path(__file__).resolve().parents[1]))\nfrom src.features import make_features, training_matrix\n\n\ndef fixture(periods=500):\n    dates = pd.date_range("2025-01-01", periods=periods)\n    records = []\n    for station in [101, 202]:\n        for hour in [6, 17]:\n            for day, date in enumerate(dates):\n                value = station + hour + day * 3\n                records.append((date, station, hour, value, value * 2))\n    history = pd.DataFrame(records, columns=["date", "station_id", "hour", "boardings", "alightings"])\n    metadata = pd.DataFrame({\n        "station_id": [101, 202], "canonical_line": [1, 2],\n        "station_name": ["A", "B"], "first_date": ["2025-01-01"]*2,\n        "last_date": ["2026-06-30"]*2,\n    })\n    return history, metadata\n\n\ndef queries(origin, horizons=(1, 7, 123)):\n    return pd.DataFrame([\n        (pd.Timestamp(origin) + pd.Timedelta(days=horizon), station, hour)\n        for horizon in horizons for station in [101, 202] for hour in [6, 17]\n    ], columns=["date", "station_id", "hour"])\n\n\nclass FeatureTests(unittest.TestCase):\n    @classmethod\n    def setUpClass(cls):\n        cls.history, cls.metadata = fixture()\n        cls.origin = pd.Timestamp("2025-12-31")\n\n    def test_future_target_and_alightings_mutation_cannot_change_features(self):\n        query = queries(self.origin)\n        expected = make_features(self.history, query, self.metadata, self.origin)\n        mutated = self.history.copy()\n        future = mutated.date > self.origin\n        mutated.loc[future, "boardings"] = 999_999_999\n        mutated.loc[future, "alightings"] = -999_999_999\n        actual = make_features(mutated, query, self.metadata, self.origin)\n        assert_frame_equal(expected, actual, check_exact=True)\n        # Supplying hidden labels on query rows must not expose them as features.\n        labelled_query = query.assign(boardings=123_456, alightings=654_321)\n        assert_frame_equal(expected, make_features(self.history, labelled_query, self.metadata, self.origin), check_exact=True)\n\n    def test_query_reordering_keeps_each_feature_attached_to_its_key(self):\n        query = queries(self.origin)\n        expected = make_features(self.history, query, self.metadata, self.origin)\n        permutation = np.array([8, 2, 11, 0, 5, 9, 3, 7, 1, 10, 4, 6])\n        actual = make_features(self.history, query.iloc[permutation], self.metadata, self.origin)\n        assert_frame_equal(expected.iloc[permutation].reset_index(drop=True), actual, check_exact=True)\n\n    def test_seasonal7_repeats_latest_observed_weekday_for_all_horizons(self):\n        query = queries(self.origin)\n        features = make_features(self.history, query, self.metadata, self.origin)\n        for index, row in query.iterrows():\n            observed = self.history.loc[\n                (self.history.station_id == row.station_id)\n                & (self.history.hour == row.hour)\n                & (self.history.date <= self.origin)\n                & (self.history.date.dt.dayofweek == row.date.dayofweek)\n            ].sort_values("date").iloc[-1]\n            self.assertEqual(features.loc[index, "seasonal7"], observed.boardings)\n            self.assertEqual(features.loc[index, "seasonal7_alight"], observed.alightings)\n            self.assertEqual(features.loc[index, "horizon"], (row.date - self.origin).days)\n\n    def test_year364_never_uses_lookup_after_origin(self):\n        origin = pd.Timestamp(\'2025-11-26\')\n        query = queries(origin, horizons=(1, 123, 364, 365, 400))\n        features = make_features(self.history, query, self.metadata, origin)\n        for index, row in query.iterrows():\n            lookup_date = row.date - pd.Timedelta(days=364)\n            if lookup_date > origin or lookup_date < self.history.date.min():\n                self.assertTrue(np.isnan(features.loc[index, "year364"]))\n            else:\n                match = self.history.loc[(self.history.date == lookup_date) & (self.history.station_id == row.station_id) & (self.history.hour == row.hour)]\n                self.assertEqual(features.loc[index, "year364"], match.boardings.iloc[0])\n\n    def test_metadata_last_date_and_future_first_date_do_not_leak(self):\n        query = queries(self.origin)\n        expected = make_features(self.history, query, self.metadata, self.origin)\n        metadata = self.metadata.assign(first_date="2099-01-01", last_date="1900-01-01", station_name="Arbitrary")\n        actual = make_features(self.history, query, metadata, self.origin)\n        assert_frame_equal(expected, actual, check_exact=True)\n        self.assertNotIn("last_date", actual.columns)\n        self.assertNotIn("first_date", actual.columns)\n\n    def test_full_123_day_label_window_cutoff_boundary(self):\n        # First anchor is Jan 28. A cutoff exactly 123 days later permits it;\n        # one day earlier must not shorten the training label window silently.\n        history, metadata = fixture(periods=200)\n        anchor = pd.Timestamp("2025-01-28")\n        cutoff = anchor + pd.Timedelta(days=123)\n        x, y, audit = training_matrix(history, metadata, cutoff, horizon=123, max_rows=10000)\n        self.assertEqual(len(audit), 1)\n        self.assertEqual(audit[0]["origin"], "2025-01-28")\n        self.assertEqual(pd.Timestamp(audit[0]["label_end"]), cutoff)\n        self.assertEqual(x.horizon.min(), 1)\n        self.assertEqual(x.horizon.max(), 123)\n        self.assertEqual(len(x), 123 * 4)\n        self.assertEqual(len(x), len(y))\n        self.assertTrue(all(pd.Timestamp(a["label_end"]) <= cutoff for a in audit))\n        self.assertTrue(all(pd.Timestamp(a["origin"]) + pd.Timedelta(days=123) <= cutoff for a in audit))\n        with self.assertRaisesRegex(ValueError, "Insufficient history"):\n            training_matrix(history, metadata, cutoff - pd.Timedelta(days=1), horizon=123)\n        # Changing labels beyond the cutoff cannot change either X or y.\n        mutated = history.copy()\n        mutated.loc[mutated.date > cutoff, ["boardings", "alightings"]] = 999_999\n        changed_x, changed_y, changed_audit = training_matrix(mutated, metadata, cutoff, horizon=123, max_rows=10000)\n        assert_frame_equal(x, changed_x, check_exact=True)\n        np.testing.assert_array_equal(y, changed_y)\n        self.assertEqual(audit, changed_audit)\n\n\nif __name__ == "__main__":\n    unittest.main()\n',
    'tests/test_holiday_calendar.py': '"""Holiday declarations must not leak across the forecast-origin boundary."""\nfrom pathlib import Path\nimport sys\nimport unittest\n\nimport numpy as np\nimport pandas as pd\n\nsys.path.insert(0, str(Path(__file__).resolve().parents[1]))\nfrom src.holiday_calendar import calendar_features, holiday_records\n\n\nclass HolidayCalendarTests(unittest.TestCase):\n    def test_temporary_holiday_announcement_boundary(self):\n        dates = ["2025-01-26", "2025-01-27", "2025-01-28"]\n        before = calendar_features(dates, as_of="2025-01-13")\n        after = calendar_features(dates, as_of="2025-01-14")\n        self.assertEqual(before.is_holiday.tolist(), [0, 0, 1])\n        self.assertEqual(after.is_holiday.tolist(), [0, 1, 1])\n        self.assertEqual(before.loc[0, "is_holiday_eve"], 0)\n        self.assertEqual(after.loc[0, "is_holiday_eve"], 1)\n        self.assertEqual(before.loc[1, "holiday_type"], 0)\n\n    def test_2025_presidential_election_was_not_known_in_january(self):\n        self.assertEqual(calendar_features(["2025-06-03"], "2025-01-31").is_holiday.iloc[0], 0)\n        after = calendar_features(["2025-06-03"], "2025-04-08")\n        self.assertEqual(after.is_holiday.iloc[0], 1)\n        self.assertEqual(after.is_election_day.iloc[0], 1)\n\n    def test_labor_day_and_constitution_day_new_designations(self):\n        dates = ["2025-05-01", "2026-05-01", "2025-07-17", "2026-07-17"]\n        final = calendar_features(dates)\n        self.assertEqual(final.is_holiday.tolist(), [0, 1, 0, 1])\n        before = calendar_features(dates, "2026-02-02")\n        self.assertEqual(before.is_holiday.tolist(), [0, 0, 0, 0])\n        self.assertEqual(before.is_labor_day.tolist(), [1, 1, 0, 0])\n\n    def test_no_false_substitutes(self):\n        dates = ["2026-09-26", "2026-09-27", "2026-09-28", "2026-06-08", "2026-10-05"]\n        frame = calendar_features(dates)\n        self.assertEqual(frame.is_holiday.tolist(), [1, 0, 0, 0, 1])\n        self.assertEqual(frame.is_rest_day.tolist(), [1, 1, 0, 0, 1])\n        self.assertEqual(frame.is_substitute_holiday.tolist(), [0, 0, 0, 0, 1])\n\n    def test_year_and_test_scope_counts(self):\n        records = holiday_records()\n        self.assertEqual((records.date.str.startswith("2025")).sum(), 19)\n        self.assertEqual((records.date.str.startswith("2026")).sum(), 22)\n        test = calendar_features(pd.date_range("2026-07-01", "2026-10-31"))\n        self.assertEqual(test.is_holiday.sum(), 9)\n\n    def test_order_index_dtype_and_empty_input(self):\n        dates = pd.Series(["2026-10-05", "2025-05-01", "2026-10-05"], index=[9, 4, 9])\n        frame = calendar_features(dates)\n        self.assertEqual(frame.index.tolist(), [9, 4, 9])\n        self.assertEqual(frame.is_holiday.tolist(), [1, 0, 1])\n        self.assertTrue(all(t == np.dtype("int8") for t in frame.dtypes))\n        self.assertEqual(calendar_features([]).shape, (0, 12))\n\n    def test_unsupported_dates_and_post_cutoff_origin_rejected(self):\n        for dates in [[None], ["2027-01-01"], ["2024-12-31"]]:\n            with self.assertRaises(ValueError):\n                calendar_features(dates)\n        with self.assertRaises(ValueError):\n            calendar_features(["2026-07-17"], "2026-07-01")\n\n\nif __name__ == "__main__":\n    unittest.main()\n',
    'tests/test_metrics_validation.py': '"""Behavior checks for score conventions, leakage and submission corruption."""\nimport math\nfrom pathlib import Path\nimport sys\nimport unittest\n\nimport numpy as np\nimport pandas as pd\n\nsys.path.insert(0, str(Path(__file__).resolve().parents[1] / "src"))\nfrom metrics import forecast_diagnostics, regression_metrics, residual_autocorrelation, score, seasonal_scales\nfrom validate_submission import validate_submission\n\n\nclass MetricTests(unittest.TestCase):\n    def test_score_conventions(self):\n        result = regression_metrics([0, 2, 4], [0, 4, 1])\n        self.assertAlmostEqual(result["mae"], 5 / 3)\n        self.assertAlmostEqual(result["rmse"], math.sqrt(13 / 3))\n        self.assertAlmostEqual(result["mbe"], -1 / 3)\n        self.assertAlmostEqual(result["smape"], (0 + 200 * 2 / 6 + 200 * 3 / 5) / 3)\n        self.assertAlmostEqual(result["rmsle"], math.sqrt(((math.log(5)-math.log(3))**2 + (math.log(2)-math.log(5))**2)/3))\n        self.assertEqual(score([0, 0], [0, 0], "smape"), 0)\n\n    def test_invalid_inputs_rejected(self):\n        for y, p in [([], []), ([1], [1, 2]), ([1], [np.nan]), ([np.inf], [1])]:\n            with self.assertRaises(ValueError):\n                regression_metrics(y, p)\n        with self.assertRaises(ValueError):\n            score([1], [-1], "rmsle")\n\n    def test_official_rmsle_is_root_mean_squared_log1p_error(self):\n        truth = np.expm1(np.array([0., 1., 2.]))\n        predictions = np.expm1(np.array([1., 3., 5.]))\n        expected = math.sqrt((1.0**2 + 2.0**2 + 3.0**2) / 3)\n        self.assertAlmostEqual(score(truth, predictions, "rmsle"), expected, places=12)\n        self.assertAlmostEqual(regression_metrics(truth, predictions)["rmsle"], expected, places=12)\n        self.assertEqual(score([0, 1, 20], [0, 1, 20], "rmsle"), 0)\n        self.assertAlmostEqual(score([0], [1], "rmsle"), math.log(2), places=12)\n\n    def test_seasonal_scale_matches_calendar_not_rows(self):\n        train = pd.DataFrame({"station_id": [1]*3, "hour": [6]*3, "date": ["2025-01-01", "2025-01-08", "2025-01-16"], "boardings": [10, 12, 99]})\n        scales = seasonal_scales(train)\n        self.assertEqual(scales.loc[0, "seasonal_pairs"], 1)\n        self.assertEqual(scales.loc[0, "mase_scale"], 2)\n\n    def test_zero_scales_undefined_and_coverage_reported(self):\n        train = pd.DataFrame({"station_id": [1]*8+[2]*8, "hour": [6]*16, "date": list(pd.date_range("2025-01-01", periods=8))*2, "boardings": list(range(8))+[3]*8})\n        valid = pd.DataFrame({"station_id": [1, 2], "hour": [6, 6], "date": ["2025-01-09"]*2, "boardings": [8, 3]})\n        result = forecast_diagnostics(train, valid, [15, 4])\n        self.assertAlmostEqual(result["summary"]["mase_macro"], 1)\n        self.assertAlmostEqual(result["summary"]["mase_row_weighted"], 1)\n        self.assertEqual(result["summary"]["mase_row_coverage"], 0.5)\n        self.assertEqual(result["summary"]["mase_undefined_series"], 1)\n        self.assertEqual(result["per_step"]["horizon"].tolist(), [1])\n        with self.assertRaisesRegex(ValueError, "leakage"):\n            forecast_diagnostics(train, valid, [15, 4], origin="2025-01-07")\n\n    def test_residual_acf_does_not_cross_groups_or_missing_days(self):\n        frame = pd.DataFrame({"station_id": [1]*4 + [2]*4, "hour": [6]*8, "date": list(pd.date_range("2025-01-01", periods=4))*2, "residual": [1,2,3,4,100,90,80,70]})\n        acf = residual_autocorrelation(frame, lags=(1, 7))\n        self.assertTrue(np.allclose(acf.loc[acf.lag_days.eq(1), "acf"], 1))\n        self.assertEqual(acf.loc[acf.lag_days.eq(1), "pairs"].tolist(), [3, 3])\n        self.assertEqual(acf.loc[acf.lag_days.eq(7), "pairs"].tolist(), [0, 0])\n        self.assertTrue(acf.loc[acf.lag_days.eq(7), "acf"].isna().all())\n\n\nclass SubmissionTests(unittest.TestCase):\n    def setUp(self):\n        self.test = pd.DataFrame({"example_id": ["J0", "J1", "J2"], "date": ["2026-07-01", "2026-09-30", "2026-10-01"]})\n        self.sample = pd.DataFrame({"example_id": ["J0", "J1"], "boardings": [0, 0]})\n        self.sub = pd.DataFrame({"example_id": ["J0", "J1"], "boardings": [1.2, 2.3]})\n\n    def test_template_discrepancy_explicit(self):\n        result = validate_submission(self.sub, self.test, self.sample, scope="template")\n        self.assertTrue(result["valid"])\n        self.assertFalse(result["template_matches_test"])\n        self.assertEqual(result["scope_discrepancy"]["test_ids_absent_from_template"], 1)\n        self.assertEqual(len(result["warnings"]), 1)\n        self.assertFalse(result["matches_official_scope"])\n        self.assertIn("diagnostic only", result["warnings"][0])\n        self.assertEqual(result["scope_discrepancy"]["missing_period_start"], "2026-10-01")\n        with self.assertRaisesRegex(ValueError, "requires 3"):\n            validate_submission(self.sub, self.test, self.sample, scope="test")\n\n    def test_invalid_files_rejected(self):\n        invalid = [self.sub.iloc[::-1], self.sub.assign(boardings=[np.nan, 2]), self.sub.assign(boardings=[np.inf, 2]), self.sub.assign(boardings=[-1, 2]), self.sub.assign(example_id=["J0", "J0"]), self.sub.assign(extra=0), self.sub.assign(example_id=["J0", "BAD"])]\n        for frame in invalid:\n            with self.subTest(frame=frame.to_dict()):\n                with self.assertRaises(ValueError):\n                    validate_submission(frame, self.test, self.sample, scope="template")\n\n    def test_test_scope_and_distribution(self):\n        sub = pd.DataFrame({"example_id": self.test.example_id, "boardings": [1, 2, 3]})\n        result = validate_submission(sub, self.test, self.sample, scope="test", reference_predictions=[1, 1, 1])\n        self.assertEqual(result["rows"], 3)\n        self.assertTrue(result["matches_official_scope"])\n        self.assertEqual(result["distribution_comparison"]["mean_ratio"], 2)\n\n    def test_default_requires_official_full_test_scope(self):\n        with self.assertRaisesRegex(ValueError, "requires 3"):\n            validate_submission(self.sub, self.test, self.sample)\n        full = pd.DataFrame({"example_id": self.test.example_id, "boardings": [1, 2, 3]})\n        result = validate_submission(full, self.test, self.sample)\n        self.assertEqual(result["scope"], "test")\n        self.assertTrue(result["matches_official_scope"])\n\n\nif __name__ == "__main__":\n    unittest.main()\n',
    'configs/default.json': '{\n  "seed": 2026,\n  "official_metric_verified": true,\n  "competition_rules_verified": true,\n  "rules_evidence": "user-supplied official Overview, Data and Rules on 2026-11-11",\n  "selection_metric": "rmsle",\n  "training_loss": "log_mse",\n  "horizon_days": 123,\n  "fold_origins": [\n    "2025-06-26",\n    "2025-10-27",\n    "2026-02-27"\n  ],\n  "fold_roles": [\n    "development",\n    "development",\n    "audit"\n  ],\n  "origin_stride_days": 28,\n  "max_training_rows": 220000,\n  "threads": 4,\n  "device": "cpu",\n  "models": [\n    "lightgbm_large",\n    "xgboost",\n    "catboost"\n  ],\n  "correlation_cutoff": 0.95,\n  "use_external_data": true,\n  "external_data_allowlist": [\n    "preannounced_korean_holidays"\n  ],\n  "use_pretrained_weights": false,\n  "ensemble_policy": {\n    "method": "primary_constrained_log_mse",\n    "primary_model": "lightgbm_large",\n    "primary_min_weight": 0.5,\n    "bias_correction": false\n  },\n  "evaluation_context": {\n    "status": "reused_validation_after_prior_diagnostics",\n    "policy_selection_has_seen_audit": true,\n    "weight_fit_uses_audit": false,\n    "note": "Model selection and removal of bias calibration follow prior fold-3 diagnostics and the user request. Weights use folds 1 and 2 only. Fold 3 is a reused comparison, not a fresh holdout."\n  }\n}\n',
    'requirements.txt': 'numpy==2.3.5\npandas==2.2.3\npolars==2.1.0\nscikit-learn==1.9.1\nscipy==1.18.1\nlightgbm==4.7.0\ncatboost==1.2.10\nxgboost==3.4.1\ntorch==2.15.0\njoblib==1.6.0\nmatplotlib==3.11.2\n',
    'reports/holiday_sources.md': '# 공휴일 외부 데이터 출처와 사용 시점\n\n조회일: **2026-11-11 (Asia/Seoul)**. 사용한 달력 사실·공식 발표는 모두 **2026-06-30 이전 공개**된 자료다. 조회일과 정보 공개일을 구별한다. 실현된 교통량·날씨·행사 결과·투표 결과는 수집하거나 사용하지 않았다.\n\n대회에서 허용한 2025-01-01~2026-10-31 공휴일 특징을 `src/holiday_calendar.py`에 정적으로 인코딩했다. 모듈은 같은 공개 달력에 있는 2026-12-25까지 지원하지만 모델의 실제 예측 범위는 10월 31일까지다. 외부 API·인터넷 없이 재현된다.\n\n## 근거 자료\n\n| 코드의 source_id | 기관·문서 | 공개일 | 채택한 사실·조건 |\n| --- | --- | --- | --- |\n| annual_2025 | [한국천문연구원에 게시된 우주항공청 2025년도 월력요항 발표](https://www.kasi.re.kr/kor/publication/post/newsMaterial/30071) | 2024-06-20 | 2025년 설·추석·부처님오신날 및 대체공휴일. 세부 날짜는 [KASI 2025 달력자료](https://astro.kasi.re.kr/kor/life/post/calendarData?search_year=2025), 2024-05-07 생성 자료와 대조. 사전 달력자료의 미확정 표시를 인지하고 최종 월력요항 발표로 확인. |\n| annual_2026 | [우주항공청 2026년 월력요항 발표](https://www.kasa.go.kr/prog/plcyBrf/brief/kor/sub01_01_04/view.do?plcyBrfNo=202) | 2025-06-30 | 2026년 기본 공휴일·대체공휴일. 본문에 설 2/16~18, 대체 3/2·5/25·8/17·10/5, 추석 9/24~26 및 9/27 일요일까지의 연휴 확인. 이 발표 이후 추가된 노동절·제헌절은 아래 공지로 보완. |\n| annual_2026 보조 확인 | [한국천문연구원 2026 달력자료](https://astro.kasi.re.kr/kor/life/post/calendarData?search_year=2026) | 2024-07-25 생성 | 기본 날짜 표와 6/3 지방선거일 확인. 이 자료는 최종 공식 월력요항 발표 전 사전자료이며 그 자체를 최신 공휴일 확정본으로 취급하지 않음. [최종 월력요항 안내](https://astro.kasi.re.kr/kor/life/post/almanac?search_year=2026)와 우주항공청 발표를 함께 사용. |\n| temporary_20250127 | [인사혁신처 1월 27일 임시공휴일 지정 국무회의 의결, 정책브리핑 전재](https://www.korea.kr/briefing/pressReleaseView.do?newsId=156670135) | 2025-01-14 | 2025-01-27 임시공휴일. [정식 공고 제2025-28호](https://www.mpm.go.kr/mpm/comm/noti/mpmNotice/?boardId=bbs_0000000000000205&category=&cntId=904&mode=view&pageIdx=8)는 1/15 게시. CV 가용일은 공개 의결 발표인 1/14. |\n| election_20250603 | [행정안전부 제21대 대통령 선거일 브리핑](https://m.korea.kr/briefing/policyBriefingView.do?newsId=156683072) | 2025-04-08 | 2025-06-03 선거일 및 임시공휴일 지정. [행안부 보도자료](https://mois.go.kr/frt/bbs/type010/commonSelectBoardArticle.do?bbsId=BBSMSTR_000000000008&nttId=116942)와 대조. |\n| annual_2026 선거 보조 확인 | [서울특별시선거관리위원회 제9회 전국동시지방선거 주요일정](https://su.nec.go.kr/su/bbs/B0000263/view.do?category1=su&category2=&deleteCd=0&menuNo=100176&nttId=273718&pageIndex=1) | 2026-01-30 | 선거일 2026-06-03 확인. 선거 결과는 열람·사용하지 않음. 날짜 자체는 KASI의 2024-07-25 달력자료에도 있어 모듈의 보수적 연간 달력 가용일 2025-06-30보다 먼저 공개됨. |\n| constitution_2026 | [인사혁신처 제헌절, 18년 만에 공휴일 재지정](https://www.mpm.go.kr/mpm/comm/newsPress/newsPressRelease/?boardId=bbs_0000000000000029&category=&cntId=4205&mode=view&pageIdx=) | 2026-02-03 | 2026-07-17부터 공휴일임을 공개 발표. [정책브리핑 전재](https://www.korea.kr/briefing/pressReleaseView.do?newsId=156742873)에서도 날짜·배포일 확인. |\n| labor_2026 | [관계부처 합동 노동절 제정 63년 만에 공휴일 지정](https://www.korea.kr/briefing/pressReleaseView.do?newsId=156753075) | 2026-04-06 | 2026-05-01부터 공휴일. 발표 전 CV origin에서는 관공서 공휴일로 표시하지 않음. |\n| 2026 신규 지정 최종 보완 | [인사혁신처 올해부터 노동절, 제헌절 공휴일로](https://www.mpm.go.kr/mpm/comm/newsPress/newsPressRelease/?boardId=bbs_0000000000000029&cntId=4250&mode=view) | 2026-04-29 게시, 4/28 의결 | 노동절·제헌절의 관공서 공휴일 편입과 대체공휴일 적용. 제헌절 법률 2/10 공포·5/11 시행, 노동절 법률 4/9 공포·5/1 시행 경과 확인. 예측에 쓰는 `known_since`는 앞선 공식 공개 발표일이다. |\n\n## 라이선스·재사용 기록\n\n- 우주항공청 **2026년 월력요항 발표** 페이지는 **공공누리 제1유형(출처표시)**를 표시한다.\n- 우주항공청 **2025년 월력요항**의 [원 게시물](https://www.kasa.go.kr/prog/bbsArticle/BBSMSTR_000000000010/view.do?bbsId=BBSMSTR_000000000010&nttId=B000000000633Wg5pO7)은 **공공누리 제2유형(출처표시·상업적 이용금지)**를 표시한다. 대회가 요구하는 비상업적 이용 범위로 출처를 남긴다.\n- 정책브리핑의 인사혁신처·고용노동부 보도자료 전재 페이지는 **텍스트에 한해 공공누리 제1유형**을 표시한다. 본 프로젝트는 날짜·명칭·공개일을 정리하며 사진·그림·서식 파일을 재배포하지 않는다.\n- 행정안전부 보도자료는 **공공누리 제1유형**을 표시한다.\n- 서울선관위 일정 안내는 **공공누리 제4유형(출처표시·상업적 이용금지·변경금지)**를 표시한다. 원 이미지·홍보물은 복제하거나 수정하지 않았으며 공개 선거일의 교차 확인에만 사용했다. 모델에 쓰는 날짜는 이전에 공개된 KASI 달력에도 있다.\n- KASI의 [공공데이터포털 특일 정보](https://www.data.go.kr/data/15012690/openapi.do)는 **이용허락범위 제한 없음**, 무료, 등록 2015-12-21·수정 2023-03-29로 표시한다. [천문우주지식정보 카탈로그](https://www.data.go.kr/catalog/2810264/fileData.json) 역시 월력요항 등을 포함하며 **이용허락범위 제한 없음**, 수정 2023-11-15를 명시한다. 이는 달력 데이터 이용 조건의 참고 기록이며 이번 작업에서 인증 API를 호출했다고 주장하지 않는다.\n- 코드는 위 달력의 사실을 직접 작성한 정적 날짜 목록으로 보관한다. 보도자료 본문이나 디자인을 통째로 복제하지 않는다. 출처 URL·기관·공개일·조회일은 이 문서에 남긴다.\n\n## 예측 시점과 Local CV\n\n기본 호출 `calendar_features(dates, as_of="2026-06-30")`은 최종 예측 시점에 공개된 달력을 적용한다. 검증 또는 과거 학습 origin을 만들 때에는 **그 origin을 as_of로 전달**한다. 임시공휴일·신규 공휴일은 다음 조건으로 활성화된다.\n\n| 대상 날짜 | 내용 | 최초 사용 가능한 공개일 |\n| --- | --- | --- |\n| 2025-01-27 | 임시공휴일 | 2025-01-14 |\n| 2025-06-03 | 대통령선거 임시공휴일 | 2025-04-08 |\n| 2026-05-01 | 노동절의 관공서 공휴일 적용 | 2026-04-06 |\n| 2026-07-17 | 제헌절 공휴일 재지정 | 2026-02-03 |\n\n공개일 전이면 해당 날짜의 `is_holiday=0`, `holiday_type=0`으로 처리한다. ‘나중에 지정될 휴일’이라는 별도 마스크를 모델에 주지 않는다. `is_labor_day`는 두 해 모두 5/1이라는 기존 달력·노동 기념일 표시다. **2025-05-01은 관공서 공휴일로 처리하지 않는다.**\n\n주말은 named holiday와 구별한다. `is_holiday`에는 정기 일요일을 넣지 않고 `calendar_is_weekend`, `is_rest_day`에서 토·일을 표현한다. `is_holiday_eve`, `is_after_holiday`, `is_bridge_day`도 같은 as_of의 달력만 이용한다. 승차량을 휴일이라는 이유로 0으로 강제하지 않는다.\n\n## 경계 조건 확인\n\n- 2025-05-05는 어린이날과 부처님오신날이 겹치지만 한 날짜 한 행으로 인코딩한다. 5/6은 대체공휴일이다.\n- 2025-10-08은 추석 대체공휴일이다.\n- 2026-05-25, 8/17, 10/5는 대체공휴일이다.\n- **2026-09-28은 공휴일이 아니다.** 공식 연휴는 추석 9/24~26과 일요일 9/27까지다.\n- 2026-06-06 토요일 현충일은 공휴일이지만 **6/8을 대체공휴일로 추가하지 않는다.**\n- 최종 달력의 날짜 수는 2025년 19일, 2026년 22일이다(정기 일요일 제외, 중복 공휴일은 한 날짜). 테스트 2026년 7~10월에는 9개 날짜가 포함된다.\n\n상세 날짜 목록은 `reports/holiday_calendar.csv`에 있다. 이 목록은 감사·재현용이며 `known_since`, `source_id`를 모델 입력으로 넣지 않는다.\n',
}
SOURCE_SHA256 = {'configs/default.json': '143782a2f08fc6a6546cd5ab80c014efd4de2f65d3c2f2901f557c797f6eb490', 'reports/holiday_sources.md': '0988a6cd6d3eda4d703e7a6250c9c38f9c267c4c9a4b05b1796082660ebc910a', 'requirements.txt': '0b72d59bf1f0bb4f67719ece9451c5ee3d3f48a040be818031ea939fcddc21a2', 'src/__init__.py': 'f526452b64e34c742e0f8ce4ab662b851774fc308d3c57f7ac247d51f302d207', 'src/eda.py': '9891970efc533f6b3d9510b3d5d49ac409511e7224d02cb7312a481bf28e54df', 'src/ensemble.py': '63540aae5782f9b0b12a55b758767ec81ff5787e2ad6199267ae7d9bd6c04ef9', 'src/features.py': '0909aed9abaed5f8f4722839352e84b2e30836ac269505b8a56b2cc8d6ea86ed', 'src/holiday_calendar.py': '50defb4fb19fcc52a1f35e568deb8d94e5b89ec275e7ace57138cff658a17e64', 'src/metrics.py': '9fdda3b049e583d372401a4ddc6066c81ae8d5b2d1ef4560af4ffd23fa78c571', 'src/models.py': '96ea5e3ad4e7a8a16625a15c42575fccb6baa85d919edab67428a967867fc1a4', 'src/pipeline.py': '0c8b20cbba544d16a2a1da5181f6a37a09beaeaf8c48cd8c91064bbae3f9d344', 'src/validate_submission.py': '64667df2339bcf0d5fc4466807dc3ef21c892980a81ed862c3f919c3aece3d4e', 'tests/test_ensemble.py': 'ba1cd92bd065ff757fcd480fa23e9f664a7075ddef3f53c3572f5957490f77f7', 'tests/test_features.py': 'd8a7966d2bc219f8230fd37822cc2893f73a47aa11fe9c24575bfd3472afddd1', 'tests/test_holiday_calendar.py': 'c9e76a9f4936f28bacb42e110a3739822782e8cdbd91a9eac8604a3372e81c48', 'tests/test_metrics_validation.py': 'ae5e956c969e3b02c55f9f6c934d86fdd19bb0e8dd009c328582fd84f5da5680'}
for relative, content in EMBEDDED_FILES.items():
    relative_path = Path(relative)
    if relative_path.is_absolute() or ".." in relative_path.parts:
        raise ValueError(f"Unsafe embedded relative path: {relative}")
    if hashlib.sha256(content.encode("utf-8")).hexdigest() != SOURCE_SHA256[relative]:
        raise ValueError(f"Embedded source checksum mismatch: {relative}")
    destination = PROJECT_ROOT / relative_path
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_text(content, encoding="utf-8")
(PROJECT_ROOT / "source_manifest.json").write_text(json.dumps(SOURCE_SHA256, indent=2), encoding="utf-8")
print(f"Restored {len(EMBEDDED_FILES)} files to {PROJECT_ROOT}")
print("Python:", sys.version)


## 2. 데이터 경로 확인

Kaggle에서는 연결된 입력 중 네 파일이 모두 있고 헤더가 일치하는 폴더를 찾습니다.
여러 후보가 있으면 임의로 선택하지 않으므로 `DATA_DIR_OVERRIDE`에 사용할 경로를 지정합니다.
Colab에서는 업로드·마운트한 CSV 폴더의 절대 경로를 입력합니다.


In [ ]:
DATA_DIR_OVERRIDE = ""  # Colab example: "/content/sincon-2026"
EXPECTED_HEADERS = {
    "train.csv": ["date", "station_id", "hour", "boardings", "alightings"],
    "test.csv": ["example_id", "date", "station_id", "hour"],
    "metaData.csv": ["station_id", "station_name", "canonical_line", "first_date", "last_date"],
    "sample_submission.csv": ["example_id", "boardings"],
}

def matches_input(directory):
    directory = Path(directory)
    for filename, expected_header in EXPECTED_HEADERS.items():
        path = directory / filename
        if not path.is_file():
            return False
        with path.open(encoding="utf-8-sig", newline="") as handle:
            if next(csv.reader(handle), None) != expected_header:
                return False
    return True

if DATA_DIR_OVERRIDE:
    DATA_DIR = Path(DATA_DIR_OVERRIDE).expanduser().resolve()
    if not matches_input(DATA_DIR):
        raise ValueError(f"Missing CSV or unexpected header in {DATA_DIR}")
elif IS_KAGGLE:
    candidates = sorted({p.parent.resolve() for p in Path("/kaggle/input").rglob("train.csv") if matches_input(p.parent)})
    if len(candidates) != 1:
        raise ValueError(f"Expected one matching dataset; found {candidates}. Set DATA_DIR_OVERRIDE explicitly.")
    DATA_DIR = candidates[0]
else:
    DATA_DIR = Path("/content/sincon-2026")
    if not matches_input(DATA_DIR):
        raise ValueError("Set DATA_DIR_OVERRIDE to the Colab folder containing all four original CSV files.")

OUT_DIR = PROJECT_ROOT / "outputs" / "three_tree"
print("Data:", DATA_DIR)
print("Output:", OUT_DIR)
print("Input byte sizes:", {name: (DATA_DIR / name).stat().st_size for name in EXPECTED_HEADERS})


## 3. 의존성 설치

참조 실행의 버전을 고정한 `requirements.txt`를 설치합니다. Kaggle에서는 이 셀을 실행할 때
인터넷을 켜야 할 수 있습니다. 오프라인 wheel 폴더가 있으면 `WHEELHOUSE`를 지정하면 됩니다.
이미 같은 버전을 준비했다면 `INSTALL_REQUIREMENTS=False`로 바꿀 수 있습니다.
설치 후 각 작업은 별도 Python 프로세스로 실행하므로 이 노트북 커널의 기존 import 상태에 의존하지 않습니다.

대회 자체의 실행시간 제한과 플랫폼 세션 제한은 구별합니다. 제공된 대회 규정에는 별도 실행시간 제한이 없으며,
[Kaggle 일반 문서](https://www.kaggle.com/docs/notebooks)의 CPU/GPU 세션은 12시간, TPU는 9시간입니다.
실제 사용 가능 RAM·GPU 및 주간 GPU 잔여 시간은 계정의 현재 표시를 확인해야 합니다.
이 파이프라인의 기본 설정은 CPU 4개 스레드이며, 실행시간과 최대 RSS를 결과에 기록합니다.


In [ ]:
INSTALL_REQUIREMENTS = True
WHEELHOUSE = ""  # Optional existing directory for offline pip wheels.
if INSTALL_REQUIREMENTS:
    command = [sys.executable, "-m", "pip", "install", "--disable-pip-version-check", "-r", str(PROJECT_ROOT / "requirements.txt")]
    if WHEELHOUSE:
        wheel_dir = Path(WHEELHOUSE)
        if not wheel_dir.is_dir():
            raise ValueError(f"Wheel directory missing: {wheel_dir}")
        command += ["--no-index", "--find-links", str(wheel_dir)]
    subprocess.run(command, check=True, cwd=PROJECT_ROOT)

version_check = """
import importlib.metadata, json
from pathlib import Path
expected = dict(line.strip().split('==', 1) for line in Path('requirements.txt').read_text().splitlines() if line.strip() and not line.lstrip().startswith('#'))
actual = {name: importlib.metadata.version(name) for name in expected}
print(json.dumps(actual, indent=2))
mismatched = {name: {'expected': version, 'actual': actual[name]} for name, version in expected.items() if actual[name].split('+', 1)[0] != version}
if mismatched:
    raise RuntimeError('Dependency versions differ from the reference: ' + json.dumps(mismatched))
"""
subprocess.run([sys.executable, "-c", version_check], check=True, cwd=PROJECT_ROOT)


## 4. 누수·지표·제출 검수 테스트

미래 타겟·하차량 변경이 특징에 영향을 주지 않는지, 123일 label window가 cutoff를 넘지 않는지,
RMSLE·계절 MASE·달력 ACF의 정의와 로그 공간 앙상블이 맞는지 검사합니다.
공식 전체 test 범위와 오래된 sample의 차이도 검사합니다.


In [ ]:
RUN_TESTS = True
if RUN_TESTS:
    subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"], check=True, cwd=PROJECT_ROOT)


## 5. 전체 데이터 EDA

원본을 보존하면서 행·열, 결측·중복·시간 규칙성, 역별 관측 기간, 분포·급등 후보,
달력 패턴과 역 간 상관을 확인합니다. 전체 학습 자료의 EDA 상관을 모델 특징으로 재사용하지 않습니다.


In [ ]:
subprocess.run([
    sys.executable, "-m", "src.eda", "--data-dir", str(DATA_DIR),
    "--out-dir", str(PROJECT_ROOT / "reports"),
], check=True, cwd=PROJECT_ROOT)
print("EDA report:", PROJECT_ROOT / "reports" / "EDA.md")


## 6. 고정 CV → 앙상블 → 최종 학습

개발 fold 2개에서 비음수 가중치 합=1, LightGBM Large 비중≥50%의 제약으로 log-MSE를 최소화합니다.
세 모델 모두 후보로 유지하고 상관 필터와 상수 편향 보정은 적용하지 않습니다.
가중치 최적화에는 세 번째 fold를 사용하지 않습니다. 모델 구성과 편향 보정 제거는 이전 검증 분석 이후 정했으므로
세 번째 fold 점수는 기존 구간 재검증이며 새로운 독립 holdout 점수가 아닙니다.

입력 특징은 각 예측 origin 이전 관측으로 고정하고, 123일 전체를 직접 예측합니다.
역사적 origin마다 공휴일의 당시 공개 여부도 반영합니다.
모델 후보는 LightGBM Large·XGBoost·CatBoost 세 개입니다. 계절 특징은 입력 특징으로 유지합니다.
사전학습 가중치는 사용하지 않습니다. 전체 테스트를 기준으로 최종 제출 파일을 생성합니다.


In [ ]:
config = json.loads((PROJECT_ROOT / "configs" / "default.json").read_text())
print(json.dumps(config, ensure_ascii=False, indent=2))
subprocess.run([
    sys.executable, "-m", "src.pipeline", "train",
    "--data-dir", str(DATA_DIR), "--config", str(PROJECT_ROOT / "configs" / "default.json"),
    "--output", str(OUT_DIR),
], check=True, cwd=PROJECT_ROOT)


## 7. 저장 모델 재로딩·재추론 확인

같은 입력과 저장 모델에서 CSV가 완전히 동일하게 재생성되는지 SHA-256으로 비교합니다.
입력 파일의 해시, 학습 종료일, 특징 순서가 변경되면 추론을 중단합니다.


In [ ]:
RECHECK_RELOADED_MODEL = True
if RECHECK_RELOADED_MODEL:
    reload_output = PROJECT_ROOT / "outputs" / "three_tree_reloaded"
    subprocess.run([
        sys.executable, "-m", "src.pipeline", "infer",
        "--data-dir", str(DATA_DIR), "--model", str(OUT_DIR / "model_bundle.joblib"),
        "--output", str(reload_output),
    ], check=True, cwd=PROJECT_ROOT)
    original_hash = hashlib.sha256((OUT_DIR / "submission.csv").read_bytes()).hexdigest()
    reloaded_hash = hashlib.sha256((reload_output / "submission.csv").read_bytes()).hexdigest()
    if original_hash != reloaded_hash:
        raise AssertionError("Reloaded model produced a different submission file.")
    print("Exact reload reproduction:", original_hash)


## 8. 최종 검수와 결과 확인

ID·순서·행/열·결측·무한대·음수를 검사한 전체 제출 파일만 복사합니다.
`template_diagnostic_DO_NOT_SUBMIT.csv`는 오래된 양식과의 비교용이므로 제출하지 않습니다.
Kaggle에서는 `/kaggle/working/submission.csv`, Colab에서는 프로젝트 폴더의 `submission.csv`가 최종 파일입니다.


In [ ]:
subprocess.run([
    sys.executable, "-m", "src.validate_submission",
    "--submission", str(OUT_DIR / "submission.csv"),
    "--test", str(DATA_DIR / "test.csv"),
    "--sample", str(DATA_DIR / "sample_submission.csv"),
    "--scope", "test", "--report", str(OUT_DIR / "submission_validation_cli.json"),
], check=True, cwd=PROJECT_ROOT)
validation = json.loads((OUT_DIR / "submission_validation_cli.json").read_text())
if not validation["valid"] or not validation["matches_official_scope"] or validation["rows"] != 157194:
    raise AssertionError("Official full-test submission validation failed.")
FINAL_SUBMISSION = Path("/kaggle/working/submission.csv") if IS_KAGGLE else PROJECT_ROOT / "submission.csv"
shutil.copyfile(OUT_DIR / "submission.csv", FINAL_SUBMISSION)
print("Final submission:", FINAL_SUBMISSION)
print("Submission SHA-256:", hashlib.sha256(FINAL_SUBMISSION.read_bytes()).hexdigest())

import pandas as pd
from IPython.display import display, FileLink
display(pd.read_csv(OUT_DIR / "cv_scores.csv"))
print("Reused validation summary (weights fit on folds 1 and 2 only):")
print(json.dumps(json.loads((OUT_DIR / "audit_summary.json").read_text()), ensure_ascii=False, indent=2))
print("Run summary:")
print(json.dumps(json.loads((OUT_DIR / "run_summary.json").read_text()), ensure_ascii=False, indent=2))
display(FileLink(str(FINAL_SUBMISSION)))
display(FileLink(str(OUT_DIR / "cv_scores.csv")))
display(FileLink(str(PROJECT_ROOT / "reports" / "EDA.md")))
